# 🚀 Optimizador Masivo de Estrategias · VectorBT + CUDA (RTX Pro 6000 · 96 GB VRAM)

Este notebook busca, entre **cientos de miles de combinaciones de parámetros**, las estrategias de trading que ganan dinero en el pasado (*In-Sample*) **y siguen ganando** en datos que nunca vieron (*Out-Of-Sample*).

**Cómo usarlo (sin saber programar):**
1. Elige en la Celda 2 el instrumento (ej. `XAU/USD`) y la temporalidad: los datos se **descargan solos de Dukascopy** y se guardan en tu Drive. Las siguientes veces solo se descarga lo nuevo. (También puedes usar tu propio CSV de velas o de ticks.)
2. Menú **Entorno de ejecución → Cambiar tipo de entorno** → elige la GPU.
3. Edita **solo la Celda 2** (nombre del archivo, dirección, etc.).
4. Menú **Entorno de ejecución → Ejecutar todas**.

| Celda | Qué hace |
|---|---|
| 1 | Instala librerías, monta Drive y muestra el hardware (RAM, GPU, VRAM) |
| 2 | ✏️ Tus parámetros |
| 3 | Descarga/actualiza los datos (caché en Drive) o lee tu CSV, convierte a `float32`, divide IS/OOS y aplica la **purga** anti-fugas |
| 4 | Simula toda la malla en la GPU (un hilo CUDA por estrategia) |
| 5 | Embudo de robustez en GPU (walk-forward, vecinos, estrés, Monte Carlo, prueba de azar) y examen final OOS con VectorBT |
| 6 | Guarda el CSV de estrategias robustas en Drive |

> ⚠️ Los resultados pasados no garantizan resultados futuros. Usa esto como herramienta de investigación, no como consejo financiero.

## 1️⃣ Instalación del entorno, detección de hardware y montaje de Google Drive

Instala `vectorbt`, `cupy-cuda12x`, `numba`, `pandas` y `numpy`. Después monta tu Drive, muestra un informe del hardware y compila un kernel CUDA de prueba en la GPU.

- La instalación **respeta las versiones de NumPy/Pandas que ya trae Colab** (elige la versión de VectorBT compatible), así que **no hace falta reiniciar el entorno**. Solo en el caso excepcional de que no exista una combinación compatible, el entorno se reinicia solo una vez: entonces vuelve a pulsar **Ejecutar todas**.
- Si aparece **"Kernels CUDA (CuPy): ✅"** y **"Modo GPU"** al final, todo está listo.
- Si vienes de una versión anterior de este notebook, usa antes **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** para empezar con librerías limpias.
- Si aparece **"Modo CPU"**, el notebook funcionará igual, pero mucho más lento: revisa que el entorno de ejecución tenga la GPU seleccionada.

In [ ]:
# ==============================================================================
# CELDA 1 · INSTALACIÓN DEL ENTORNO, DETECCIÓN DE HARDWARE Y GOOGLE DRIVE
# ==============================================================================
# Esta celda:
#   1) Instala las librerías necesarias (solo la primera vez, en silencio).
#      Se respetan las versiones de NumPy/Pandas de Colab (no hace falta reiniciar).
#      Solo si fuera imposible, el entorno se reinicia solo una vez al terminar de instalar:
#      en ese caso pulsa otra vez "Ejecutar todas".
#   2) Monta tu Google Drive en /content/drive.
#   3) Muestra un informe del hardware: RAM, CPU y GPU NVIDIA (con su VRAM).
#   4) Comprueba que la GPU puede compilar kernels CUDA propios (CuPy RawKernel).
#   5) Crea (si no existe) la carpeta /content/drive/MyDrive/activos y lista los CSV
#      y los datos ya descargados de internet (subcarpeta datos_descargados).
# ==============================================================================
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
import time
import warnings

warnings.filterwarnings("ignore")


class ErrorAmigable(Exception):
    """Error con mensaje en español, sin el traceback técnico (IPython muestra solo el texto)."""

    def _render_traceback_(self):
        return ["", str(self), ""]


# ¿Estamos dentro de Google Colab? (si no, el notebook funciona igual en local)
EN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

PAQUETES = [
    "vectorbt",          # Motor de backtesting vectorizado (trae NumPy, Pandas y Numba recientes)
    "cupy-cuda12x",      # NumPy sobre la GPU + compilador de kernels CUDA (RTX Pro 6000 / Blackwell)
    "numba",             # Compilador JIT (CPU) que usa VectorBT
    "pandas",
    "numpy",
    "dukascopy-python",  # Descarga de datos históricos de Dukascopy por internet
    "pyarrow",           # Formato Parquet (guardado rápido y compacto en Drive)
    "psutil",            # Lectura de RAM / CPU
    "matplotlib",        # Gráficos (mapa de calor)
]
# Módulo de Python → paquete de pip, para saber si ya está todo instalado
MODULOS_NECESARIOS = {"vectorbt": "vectorbt", "cupy": "cupy-cuda12x", "numba": "numba",
                      "dukascopy_python": "dukascopy-python", "pyarrow": "pyarrow",
                      "psutil": "psutil", "matplotlib": "matplotlib"}
# Librerías compiladas que NO se pueden cambiar de versión con el entorno en marcha
MODULOS_BINARIOS = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "numba": "numba",
                    "llvmlite": "llvmlite", "pyarrow": "pyarrow", "matplotlib": "matplotlib",
                    "sklearn": "scikit-learn", "cupy": "cupy-cuda12x"}


def versiones_en_disco():
    """Versión instalada de cada paquete. Si hay copias duplicadas, vale la primera del
    sys.path (la misma que cargaría Python con `import`)."""
    versiones = {}
    for d in importlib.metadata.distributions():
        versiones.setdefault((d.metadata["Name"] or "").lower().replace("_", "-"), d.version)
    return versiones


def modulos_desfasados(versiones):
    """Librerías cargadas en memoria con una versión distinta a la instalada en disco."""
    desfasados = []
    for modulo, paquete in MODULOS_BINARIOS.items():
        cargado = sys.modules.get(modulo)
        en_memoria = getattr(cargado, "__version__", None) if cargado else None
        en_disco = versiones.get(paquete)
        if en_memoria and en_disco and en_memoria != en_disco:
            desfasados.append(f"{modulo} {en_memoria} → {en_disco}")
    return desfasados


MARCA_REINICIO = "/tmp/.optimizador_vbt_reinicios"   # /tmp sobrevive al reinicio del kernel


def reiniciar_entorno(motivo):
    reinicios = int(open(MARCA_REINICIO).read() or 0) if os.path.exists(MARCA_REINICIO) else 0
    if reinicios >= 2:   # Nunca entrar en un bucle de reinicios
        raise ErrorAmigable(
            f"❌ Las librerías siguen sin cargar bien tras reiniciar ({motivo}).\n"
            "   👉 Entorno de ejecución → Desconectar y eliminar entorno de ejecución, "
            "y después Ejecutar todas.")
    with open(MARCA_REINICIO, "w") as f:
        f.write(str(reinicios + 1))
    print("\n" + "=" * 70)
    print("🔄 REINICIO NECESARIO (solo ocurre tras instalar librerías)")
    print(f"   Motivo: {motivo}")
    print("   El entorno se reinicia ahora mismo. Cuando termine, pulsa de nuevo:")
    print("   👉 Entorno de ejecución → Ejecutar todas")
    print("   (Tus datos de Drive no se pierden y la instalación no se repetirá.)")
    print("=" * 70, flush=True)
    time.sleep(2)
    os.kill(os.getpid(), 9)   # Colab vuelve a arrancar el entorno automáticamente


def preparar_librerias():
    antes = versiones_en_disco()
    faltan = [m for m in MODULOS_NECESARIOS if importlib.util.find_spec(m) is None]
    if faltan:
        print(f"📦 Instalando librerías (1-3 minutos la primera vez). Faltan: {', '.join(faltan)}")
        # Se FIJAN las versiones de NumPy/Pandas/SciPy/Numba que ya trae Colab: así pip elige
        # una versión de VectorBT compatible con ellas y NO hace falta reiniciar el entorno.
        fijadas = [f"{p}=={antes[p]}" for p in ("numpy", "pandas", "scipy", "numba", "llvmlite") if p in antes]
        fijadas.append("plotly<6")     # las versiones de VectorBT para Pandas 2 no funcionan con Plotly 6+
        restricciones = "/tmp/restricciones_optimizador.txt"
        with open(restricciones, "w") as f:
            f.write("\n".join(fijadas))
        resultado = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES, "-c", restricciones],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        if resultado.returncode != 0:      # Sin combinación compatible: versiones nuevas (+ reinicio)
            print("   ℹ️ No hay versión compatible con las librerías de Colab: se instalan versiones nuevas.")
            resultado = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES],
                                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        if resultado.returncode != 0:      # Si falla en bloque, se intenta uno a uno
            for paquete in PAQUETES:
                r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
                estado = "✅" if r.returncode == 0 else f"⚠️ falló: {(r.stderr or '').strip()[-200:]}"
                print(f"   {estado} {paquete}")
        importlib.invalidate_caches()
        siguen_faltando = [MODULOS_NECESARIOS[m] for m in MODULOS_NECESARIOS if importlib.util.find_spec(m) is None]
        if siguen_faltando:
            raise ErrorAmigable(f"❌ No se pudieron instalar: {', '.join(siguen_faltando)}. "
                                "Revisa tu conexión y vuelve a ejecutar esta celda.")
        print("   ✅ Librerías instaladas")
    else:
        print("✅ Librerías ya instaladas (no se reinstala nada)")

    despues = versiones_en_disco()
    cambiadas = sorted(n for n in MODULOS_BINARIOS.values()
                       if n in antes and n in despues and antes[n] != despues[n])
    desfasados = modulos_desfasados(despues)
    if (cambiadas or desfasados) and EN_COLAB:
        reiniciar_entorno("se actualizaron " + ", ".join(cambiadas or desfasados))
    elif desfasados:
        raise ErrorAmigable("❌ Hay librerías actualizadas que necesitan reiniciar el kernel: "
                            + ", ".join(desfasados) + ". Reinicia el kernel y vuelve a ejecutar.")


preparar_librerias()

try:   # Comprobación temprana: si falla aquí, el problema es la instalación
    import numpy as np
    import vectorbt as vbt
except ImportError as error:
    if EN_COLAB:
        reiniciar_entorno(f"error al importar ({str(error)[:120]})")
    raise ErrorAmigable(f"❌ No se pudo importar VectorBT: {error}") from None
if os.path.exists(MARCA_REINICIO):
    os.remove(MARCA_REINICIO)
print(f"   NumPy {np.__version__} | Pandas {sys.modules['pandas'].__version__} | VectorBT {vbt.__version__}")

# ------------------------------------------------------------------------------
# 2) Montaje de Google Drive
# ------------------------------------------------------------------------------
if EN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUTA_ACTIVOS = "/content/drive/MyDrive/activos"
    except Exception as error:
        raise ErrorAmigable(
            "❌ No se pudo montar Google Drive. Vuelve a ejecutar la celda y acepta los "
            f"permisos de acceso en la ventana emergente. Detalle técnico: {error}"
        ) from None
else:
    RUTA_ACTIVOS = os.path.abspath("activos")

# ------------------------------------------------------------------------------
# 3) Informe de hardware
# ------------------------------------------------------------------------------
import psutil

GiB = 1024 ** 3
ram = psutil.virtual_memory()
print("\n" + "=" * 70)
print("🖥️  INFORME DE HARDWARE")
print("=" * 70)
print(f"RAM del sistema     : {ram.total / GiB:,.1f} GB totales | {ram.available / GiB:,.1f} GB libres")
print(f"Núcleos de CPU      : {psutil.cpu_count(logical=True)} lógicos")

GPU_DISPONIBLE = False   # ¿Hay una GPU NVIDIA usable con CuPy?
KERNEL_CUDA_OK = False   # ¿CuPy puede compilar y lanzar kernels CUDA propios en esa GPU?
VRAM_TOTAL_GB = 0.0

try:
    import cupy as cp
    n_gpus = cp.cuda.runtime.getDeviceCount()
    if n_gpus == 0:
        raise RuntimeError("CUDA no detecta ninguna GPU.")
    props = cp.cuda.runtime.getDeviceProperties(0)
    nombre_gpu = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    vram_libre, vram_total = cp.cuda.runtime.memGetInfo()
    VRAM_TOTAL_GB = vram_total / GiB
    # Prueba real: operación en la GPU (si falla, la GPU no es usable)
    _ = float(cp.arange(1_000_000, dtype=cp.float32).sum())
    GPU_DISPONIBLE = True
    print(f"GPU detectada       : {nombre_gpu} (x{n_gpus})")
    print(f"VRAM                : {vram_total / GiB:,.1f} GB totales | {vram_libre / GiB:,.1f} GB libres")
    print(f"Compute capability  : {props['major']}.{props['minor']}")
    print(f"Multiprocesadores   : {props['multiProcessorCount']}")
except Exception as error:
    print(f"⚠️ GPU no disponible con CuPy ({error}).")
    print("   En Colab: menú 'Entorno de ejecución' → 'Cambiar tipo de entorno' → elige la GPU.")
    print("   El notebook seguirá funcionando en CPU (mucho más lento).")

# 4) Prueba de compilación de un kernel CUDA propio (el mismo mecanismo que usa la Celda 4)
if GPU_DISPONIBLE:
    try:
        _kernel_prueba = cp.RawKernel(r"""
        extern "C" __global__ void duplicar(double* x, const long long n) {
            long long i = (long long)blockDim.x * blockIdx.x + threadIdx.x;
            if (i < n) x[i] = x[i] * 2.0 + sqrt(0.0);
        }""", "duplicar")
        _x = cp.ones(1024, dtype=cp.float64)
        _kernel_prueba((4,), (256,), (_x, np.int64(_x.size)))
        cp.cuda.Stream.null.synchronize()
        if float(_x.sum()) != 2048.0:
            raise RuntimeError("resultado incorrecto en el kernel de prueba")
        KERNEL_CUDA_OK = True
        print("Kernels CUDA (CuPy) : ✅ compilados y ejecutados en la GPU")
        del _x
    except Exception as error:
        print(f"Kernels CUDA (CuPy) : ⚠️ no disponibles ({str(error)[:200]})")
        print("   Se usará Numba en CPU paralelo para la simulación.")

try:
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version",
                           "--format=csv"], capture_output=True, text=True, timeout=20)
    if _smi.returncode == 0:
        print("\n--- nvidia-smi ---\n" + _smi.stdout.strip())
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

# ------------------------------------------------------------------------------
# 5) Carpeta de activos y listado de CSV
# ------------------------------------------------------------------------------
RUTA_DATOS = os.path.join(RUTA_ACTIVOS, "datos_descargados")   # caché de descargas en Drive
try:
    os.makedirs(RUTA_DATOS, exist_ok=True)
    archivos_csv = sorted(f for f in os.listdir(RUTA_ACTIVOS)
                          if f.lower().endswith(".csv") and not f.startswith("resultados_"))
    print("\n" + "=" * 70)
    print(f"📁 Carpeta de activos: {RUTA_ACTIVOS}")
    if archivos_csv:
        for f in archivos_csv:
            tam_mb = os.path.getsize(os.path.join(RUTA_ACTIVOS, f)) / 1024 ** 2
            print(f"   • {f}  ({tam_mb:,.1f} MB)")
    else:
        print("   (sin CSV propios)")
    descargados = sorted(f for f in os.listdir(RUTA_DATOS) if f.endswith(".parquet"))
    print(f"📥 Datos descargados de internet ({RUTA_DATOS}):")
    for f in descargados:
        tam_mb = os.path.getsize(os.path.join(RUTA_DATOS, f)) / 1024 ** 2
        print(f"   • {f}  ({tam_mb:,.1f} MB)")
    if not descargados:
        print("   (ninguno todavía: se descargarán en la Celda 3)")
except Exception as error:
    raise ErrorAmigable(f"❌ No se pudo acceder a la carpeta de activos '{RUTA_ACTIVOS}': {error}") from None

print("\n✅ Entorno listo." + ("  🚀 Modo GPU." if GPU_DISPONIBLE else "  🐢 Modo CPU."))

## 2️⃣ Configuración de parámetros (✏️ la única celda que debes editar)

- **`FUENTE_DATOS`**: `"DESCARGAR"` (Dukascopy por internet) o `"CSV"` (tu archivo).
- **`INSTRUMENTO`**, **`FECHA_INICIO_DESCARGA`**, **`TEMPORALIDAD`**, **`LADO_PRECIO`**: qué descargar y en qué velas (`M1` … `D1`).
- **`NOMBRE_ARCHIVO`**: solo para `"CSV"`; nombre exacto del archivo dentro de `MyDrive/activos` (cópialo de la lista que imprime la Celda 1).
- **`DIRECCION_OPERACION`**: `"COMPRAS_SOLO"` o `"VENTAS_SOLO"`.
- **`PORCENTAJE_IS`**: parte de los datos usada para optimizar (el resto se reserva para validar).
- **`COMISION_PCT`**: coste por operación (comisión + slippage).
- **`ROBUSTEZ`**: exigencia de las pruebas de robustez y nº de simulaciones de la GPU (Monte Carlo y prueba de azar). Más simulaciones = más tiempo de proceso y resultados más fiables.
- **`GRAN_MALLE_PARAMETROS`**: rangos de las medias móviles y del RSI. Cada combinación se prueba en paralelo en la GPU; con 96 GB de VRAM puedes subir a **millones de combinaciones** (la memoria que se usa por combinación es de solo ~64 bytes).

> 💡 Sobre `vbt.Param`: es una función de **VectorBT PRO** (de pago). Con la versión gratuita de VectorBT la malla se construye con un producto cartesiano de NumPy, que es exactamente lo mismo.

In [ ]:
# ==============================================================================
# CELDA 2 · PARÁMETROS DEL USUARIO  (✏️ SOLO MODIFICA ESTA CELDA)
# ==============================================================================
import numpy as np
import pandas as pd

# 1) ORIGEN DE LOS DATOS
#    "DESCARGAR" → se descargan de Dukascopy por internet y se guardan en Drive
#                  (activos/datos_descargados). Las siguientes veces SOLO se descarga lo
#                  nuevo desde la última vela guardada.
#    "CSV"       → se usa un archivo tuyo de la carpeta activos (velas OHLC o ticks bid/ask).
FUENTE_DATOS = "DESCARGAR"

# Instrumento de Dukascopy (solo para "DESCARGAR"). Ejemplos:
#   Forex: "EUR/USD", "GBP/USD", "USD/JPY" · Metales: "XAU/USD", "XAG/USD"
#   Cripto: "BTC/USD", "ETH/USD" · Índices: "E_NQ-100" (Nasdaq 100), "E_SandP-500", "E_DAAX"
INSTRUMENTO = "XAU/USD"
FECHA_INICIO_DESCARGA = "2019-01-01"   # Desde cuándo descargar (AAAA-MM-DD)

# Temporalidad de las velas: "M1", "M5", "M10", "M15", "M30", "H1", "H4" o "D1"
#   (en "CSV", los ticks o velas más pequeñas se agrupan automáticamente en esta temporalidad)
TEMPORALIDAD = "H1"
LADO_PRECIO = "BID"                    # "BID" o "ASK"

# Archivo propio (solo para "CSV"): nombre EXACTO de la lista que imprimió la Celda 1
NOMBRE_ARCHIVO = "XAUUSD_AXI_M1.csv"

# 2) Dirección operativa: "COMPRAS_SOLO" (solo largos) o "VENTAS_SOLO" (solo cortos)
DIRECCION_OPERACION = "COMPRAS_SOLO"

# 3) Fracción de datos para In-Sample (optimización). El resto (30%) es Out-Of-Sample.
PORCENTAJE_IS = 0.70

# 4) Comisión + slippage por operación (0.00005 = 0.005% del valor negociado)
COMISION_PCT = 0.00005

# Capital inicial ficticio (solo afecta a la escala, no a los porcentajes)
CAPITAL_INICIAL = 10_000.0

# 5) GRAN MALLA DE PARÁMETROS
#    Estrategia: cruce de Media Móvil Simple rápida sobre lenta + filtro RSI.
#      • COMPRAS: entra cuando la MA rápida cruza HACIA ARRIBA la lenta y RSI < umbral
#                 (no compra en sobrecompra). Sale con el cruce hacia abajo.
#      • VENTAS : entra cuando la MA rápida cruza HACIA ABAJO la lenta y RSI > 100 - umbral
#                 (no vende en sobreventa). Sale con el cruce hacia arriba.
#      • Un umbral de 100 equivale a "sin filtro RSI".
#    Los periodos se miden en VELAS (en un archivo M1, 60 = 1 hora).
#    Con 96 GB de VRAM puedes ampliar estos rangos sin miedo: cada combinación es un
#    hilo de la GPU. Por defecto: ~126.000 combinaciones.
GRAN_MALLE_PARAMETROS = {
    "ma_rapida":   np.arange(2, 101, 2),      # 50 periodos: 2, 4, ..., 100
    "ma_lenta":    np.arange(20, 401, 5),     # 77 periodos: 20, 25, ..., 400
    "rsi_periodo": [7, 14, 21, 28],           # 4 periodos de RSI
    "rsi_umbral":  [55, 60, 65, 70, 75, 80, 85, 90, 100],  # 9 umbrales (100 = sin filtro RSI)
}

# Filtros de descarte In-Sample (eliminan estrategias perdedoras o mediocres)
FILTROS_DESCARTE = {
    "retorno_min":       0.0,    # Retorno total debe ser > 0
    "profit_factor_min": 1.2,    # Profit Factor mínimo
    "trades_min":        30,     # Mínimo de operaciones cerradas (significancia estadística)
    "trades_max":        5_000,  # Máximo de operaciones (evita sobre-operar)
    "drawdown_max":      0.25,   # Máximo drawdown permitido (0.25 = 25%)
}

# Métrica para ordenar a las supervivientes: "sharpe", "retorno_total", "profit_factor" o "calmar"
METRICA_RANKING = "sharpe"

# 6) PRUEBAS DE ROBUSTEZ (todas usan SOLO datos In-Sample; el OOS queda como examen final)
#    Aquí es donde la GPU trabaja de verdad: SUBE las simulaciones para más rigor (más tiempo).
ROBUSTEZ = {
    # Consistencia temporal: el In-Sample se parte en N tramos seguidos y la estrategia
    # debe ganar dinero en al menos M de ellos (no vale ganar solo en una racha).
    "tramos_walk_forward":          6,
    "tramos_ganadores_min":         4,
    # Estabilidad de parámetros: el Sharpe mediano de los parámetros VECINOS (un paso arriba
    # o abajo en cada parámetro) debe ser al menos esta fracción del Sharpe propio.
    "fraccion_sharpe_vecinos":      0.5,
    # Cuántas candidatas (las mejores tras los filtros anteriores) pasan a las pruebas de estrés
    "candidatas_estres":            2_000,
    # Estrés de ejecución: debe seguir ganando con costes multiplicados y entrando más tarde
    "multiplicador_costes":         2.0,
    "velas_retraso_extra":          1,
    # Monte Carlo con ruido: versiones alternativas de la historia con ruido en los precios
    "simulaciones_ruido":           200,     # ↑ más simulaciones = resultado más fiable
    "ruido_fraccion_volatilidad":   0.25,    # ruido = 25% del movimiento típico de una vela
    "ruido_ganadoras_min":          0.80,    # debe ganar en ≥ 80% de las simulaciones
    # Prueba de azar (Reality Check): la malla COMPLETA se repite sobre precios barajados y
    # sin tendencia (sin patrones reales). La estrategia, medida también sin tendencia, debe
    # superar al mejor resultado obtenido por pura suerte.
    "simulaciones_azar":            100,     # ↑ más simulaciones = umbral de suerte más preciso
    "confianza_azar":               0.99,    # 0.99 → menos de un 1% de probabilidad de ser suerte
    # Examen final OOS: además de ganar, superar el Sharpe de comprar y mantener (o vender y
    # mantener en VENTAS_SOLO) en el mismo periodo.
    "exigir_superar_buy_hold":      True,
}
TOP_N = 50   # Cuántas de las más robustas pasan al examen final Out-Of-Sample

# Criterios que debe cumplir una estrategia en Out-Of-Sample para considerarse robusta
FILTROS_OOS = {
    "retorno_min":       0.0,    # Debe seguir ganando dinero en datos nunca vistos
    "profit_factor_min": 1.0,    # Y ganar más de lo que pierde
    "trades_min":        5,      # Con un mínimo de operaciones
}

# Opciones avanzadas (normalmente no hace falta tocarlas)
ELIMINAR_VELAS_SIN_VOLUMEN = True   # Quita las velas "planas" de fin de semana de Dukascopy
FRACCION_MAX_VRAM = 0.80            # Nunca usar más del 80% de la VRAM (no saturar la VM)
LOTE_MAX_COMBINACIONES = 2_000_000  # Combinaciones por lanzamiento de kernel
MOSTRAR_GRAFICOS = True             # Mapa de calor + curva de capital

# Temporalidad → (código de la API de Dukascopy, regla de agrupación de pandas)
TEMPORALIDADES = {"M1": ("1MIN", "1min"), "M5": ("5MIN", "5min"), "M10": ("10MIN", "10min"),
                  "M15": ("15MIN", "15min"), "M30": ("30MIN", "30min"), "H1": ("1HOUR", "1h"),
                  "H4": ("4HOUR", "4h"), "D1": ("1DAY", "1D")}

# ------------------------------------------------------------------------------
# Validación de los parámetros (mensajes de error amigables)
# ------------------------------------------------------------------------------
try:
    if FUENTE_DATOS not in ("DESCARGAR", "CSV"):
        raise ErrorAmigable('❌ FUENTE_DATOS debe ser "DESCARGAR" o "CSV".')
    if TEMPORALIDAD not in TEMPORALIDADES:
        raise ErrorAmigable(f"❌ TEMPORALIDAD debe ser una de: {', '.join(TEMPORALIDADES)}.")
    if LADO_PRECIO not in ("BID", "ASK"):
        raise ErrorAmigable('❌ LADO_PRECIO debe ser "BID" o "ASK".')
    try:
        pd.Timestamp(FECHA_INICIO_DESCARGA)
    except Exception:
        raise ErrorAmigable("❌ FECHA_INICIO_DESCARGA debe tener el formato AAAA-MM-DD (ej. 2019-01-01).") from None
    if DIRECCION_OPERACION not in ("COMPRAS_SOLO", "VENTAS_SOLO"):
        raise ErrorAmigable('❌ DIRECCION_OPERACION debe ser exactamente "COMPRAS_SOLO" o "VENTAS_SOLO".')
    if not 0.1 <= PORCENTAJE_IS <= 0.95:
        raise ErrorAmigable("❌ PORCENTAJE_IS debe estar entre 0.10 y 0.95 (por ejemplo 0.70).")
    if not 0 <= COMISION_PCT < 0.05:
        raise ErrorAmigable("❌ COMISION_PCT debe ser una fracción pequeña (ej. 0.00005).")
    if METRICA_RANKING not in ("sharpe", "retorno_total", "profit_factor", "calmar"):
        raise ErrorAmigable('❌ METRICA_RANKING debe ser "sharpe", "retorno_total", "profit_factor" o "calmar".')
    for clave in ("ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"):
        valores = np.asarray(GRAN_MALLE_PARAMETROS[clave])
        if valores.size == 0:
            raise ErrorAmigable(f"❌ La lista '{clave}' de GRAN_MALLE_PARAMETROS está vacía.")
        if clave != "rsi_umbral" and ((valores < 2).any() or (valores != np.round(valores)).any()):
            raise ErrorAmigable(f"❌ Los periodos de '{clave}' deben ser números enteros >= 2.")
    if not 1 <= ROBUSTEZ["tramos_ganadores_min"] <= ROBUSTEZ["tramos_walk_forward"]:
        raise ErrorAmigable("❌ ROBUSTEZ: 'tramos_ganadores_min' debe estar entre 1 y 'tramos_walk_forward'.")
    if not 0 < ROBUSTEZ["confianza_azar"] < 1 or not 0 <= ROBUSTEZ["ruido_ganadoras_min"] <= 1:
        raise ErrorAmigable("❌ ROBUSTEZ: 'confianza_azar' y 'ruido_ganadoras_min' deben estar entre 0 y 1.")
    _umbrales = np.asarray(GRAN_MALLE_PARAMETROS["rsi_umbral"])
    if ((_umbrales <= 50) | (_umbrales > 100)).any():
        raise ErrorAmigable("❌ Los umbrales de RSI deben estar en el rango (50, 100].")
except KeyError as error:
    raise ErrorAmigable(f"❌ Falta la clave {error} en GRAN_MALLE_PARAMETROS o ROBUSTEZ.") from None

_r = np.asarray(GRAN_MALLE_PARAMETROS["ma_rapida"])
_l = np.asarray(GRAN_MALLE_PARAMETROS["ma_lenta"])
_pares_validos = int((_r[:, None] < _l[None, :]).sum())   # solo rápida < lenta
TOTAL_COMBINACIONES = (_pares_validos * len(GRAN_MALLE_PARAMETROS["rsi_periodo"])
                       * len(GRAN_MALLE_PARAMETROS["rsi_umbral"]))

print("✅ Parámetros cargados")
if FUENTE_DATOS == "DESCARGAR":
    print(f"   Datos        : descarga Dukascopy · {INSTRUMENTO} · {TEMPORALIDAD} · {LADO_PRECIO} · desde {FECHA_INICIO_DESCARGA}")
else:
    print(f"   Datos        : archivo {NOMBRE_ARCHIVO} · velas {TEMPORALIDAD}")
print(f"   Dirección    : {DIRECCION_OPERACION}")
print(f"   IS / OOS     : {PORCENTAJE_IS:.0%} / {1 - PORCENTAJE_IS:.0%}")
print(f"   Comisión     : {COMISION_PCT:.5%} por operación")
print(f"   Combinaciones: {TOTAL_COMBINACIONES:,} (pares MA válidos rápida<lenta: {_pares_validos:,})")

## 3️⃣ Descarga de datos, carga optimizada y prevención de fugas (*anti-data leakage*)

**Descarga incremental (`FUENTE_DATOS = "DESCARGAR"`):**
- La primera vez descarga todo el histórico desde Dukascopy y lo guarda en `MyDrive/activos/datos_descargados/` (formato Parquet, rápido y compacto).
- Las siguientes veces lee lo guardado y **solo descarga desde la última vela** hasta hoy.
- Guarda tras cada bloque: si se corta internet, vuelve a ejecutar la celda y continúa donde se quedó.
- Si adelantas `FECHA_INICIO_DESCARGA`, descarga solo el tramo antiguo que falta.

**Con tu propio CSV (`FUENTE_DATOS = "CSV"`):** acepta velas OHLC o **ticks** (`bid`/`ask`); los ticks y las velas más pequeñas se agrupan automáticamente en la `TEMPORALIDAD` elegida.

**Después, en ambos casos:**
1. Lee el CSV (Dukascopy `Gmt time`/`Local time`, MetaTrader 5 `<DATE> <TIME>…`, MetaTrader 4 sin cabecera o genérico) y convierte los precios a **`float32`** (la mitad de memoria que `float64`).
2. Ordena cronológicamente (del más antiguo al más reciente) y elimina duplicados y velas sin volumen.
3. Divide en **`df_is`** (entrenamiento) y **`df_oos`** (validación).
4. **Purga:** los indicadores del OOS se calculan solo con datos OOS. Sus primeras velas (tantas como el periodo del indicador más largo) no tienen indicadores fiables, así que **se eliminan** del OOS y se usan solo como calentamiento. Así ningún dato del IS contamina el OOS ni al revés.

```
|────────── In-Sample (70%) ──────────|░ purga ░|──── Out-Of-Sample ────|
```

In [ ]:
# ==============================================================================
# CELDA 3 · DESCARGA/ACTUALIZACIÓN DE DATOS + CARGA OPTIMIZADA + DIVISIÓN IS/OOS + PURGA
# ==============================================================================
#   A) Obtiene los datos: descarga incremental desde Dukascopy (guardada en Drive) o tu CSV.
#   B) Limpia y convierte a float32, ordena cronológicamente.
#   C) Divide en In-Sample / Out-Of-Sample y aplica la purga anti-fugas.
# ==============================================================================
import csv
import gc
import json
import logging
import os
import re
import time
from datetime import timezone

import numpy as np
import pandas as pd


def leer_primeras_lineas(ruta, n=3):
    with open(ruta, "r", encoding="utf-8-sig", errors="ignore") as f:
        return [f.readline().rstrip("\n") for _ in range(n)]


def detectar_separador(ruta):
    """Detecta si el CSV usa ',', ';' o tabulador como separador."""
    with open(ruta, "r", encoding="utf-8-sig", errors="ignore") as f:
        muestra = f.read(8192)
    try:
        return csv.Sniffer().sniff(muestra, delimiters=",;\t").delimiter
    except csv.Error:
        return "\t" if muestra.count("\t") > muestra.count(",") else ","


FORMATOS_FECHA = [
    "%d.%m.%Y %H:%M:%S.%f", "%d.%m.%Y %H:%M:%S", "%d.%m.%Y %H:%M",      # Dukascopy
    "%Y.%m.%d %H:%M:%S.%f", "%Y.%m.%d %H:%M:%S", "%Y.%m.%d %H:%M", "%Y.%m.%d",  # MetaTrader 4/5
    "%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d",
    "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%m/%d/%Y %H:%M:%S", "%m/%d/%Y %H:%M",
    "%Y%m%d %H:%M:%S.%f", "%Y%m%d %H:%M:%S", "%Y%m%d %H%M%S", "%Y%m%d %H:%M", "%d-%m-%Y %H:%M:%S", "%d-%m-%Y %H:%M",
]


def parsear_fechas(serie):
    """Convierte la columna de fecha probando los formatos de Dukascopy, MetaTrader, etc."""
    if pd.api.types.is_numeric_dtype(serie):              # Marca de tiempo Unix
        unidad = "ms" if serie.dropna().abs().median() > 1e11 else "s"
        return pd.to_datetime(serie, unit=unidad, errors="coerce")
    texto = serie.astype(str).str.strip()
    zona = texto.str.extract(r"GMT([+-]\d{2})(\d{2})\s*$")  # Dukascopy "Local time"
    texto = texto.str.replace(r"\s*GMT[+-]\d{4}\s*$", "", regex=True)

    # Elegimos el formato con una muestra: el que lea todas las fechas y las deje en orden
    muestra = texto.iloc[:2000]
    mejor, mejor_puntuacion = None, (0.0, 0.0)
    for formato in FORMATOS_FECHA:
        prueba = pd.to_datetime(muestra, format=formato, errors="coerce")
        leidas = prueba.notna().mean()
        if leidas < 0.99:
            continue
        ordenadas = (prueba.dropna().diff().dropna() > pd.Timedelta(0)).mean() if prueba.notna().sum() > 1 else 0
        if (leidas, ordenadas) > mejor_puntuacion:
            mejor, mejor_puntuacion = formato, (leidas, ordenadas)
    if mejor is not None:
        fechas = pd.to_datetime(texto, format=mejor, errors="coerce")
    else:
        fechas = pd.to_datetime(texto, errors="coerce")
        if fechas.isna().mean() > 0.5:
            fechas = pd.to_datetime(texto, dayfirst=True, errors="coerce")
    if zona[0].notna().any():                              # Convertir a UTC si hay desfase
        horas = zona[0].astype(float)
        minutos = zona[1].astype(float) * np.sign(horas)
        fechas = fechas - pd.to_timedelta(horas.fillna(0), unit="h") - pd.to_timedelta(minutos.fillna(0), unit="m")
    return fechas


def parece_fecha(texto):
    texto = texto.strip().strip('"')
    empieza_por_anio = len(texto) >= 6 and texto[:4].isdigit()
    empieza_por_dia = len(texto) >= 8 and texto[:2].isdigit() and texto[2] in "./-"
    return empieza_por_anio or empieza_por_dia


def cargar_csv_ohlc(ruta):
    """Lee CSV de Dukascopy, MetaTrader 4 (sin cabecera), MetaTrader 5 (<DATE> <TIME>...) o genéricos."""
    if not os.path.isfile(ruta):
        carpeta = os.path.dirname(ruta)
        disponibles = sorted(f for f in os.listdir(carpeta) if f.lower().endswith(".csv")
                             and not f.startswith("resultados_")) if os.path.isdir(carpeta) else []
        raise ErrorAmigable(
            f"❌ No se encontró '{os.path.basename(ruta)}' en {carpeta}.\n"
            f"   Archivos CSV disponibles: {disponibles or 'ninguno'}.\n"
            "   👉 Copia uno de esos nombres en NOMBRE_ARCHIVO (Celda 2), ejecuta la Celda 2 y vuelve a ejecutar esta."
        )
    sep = detectar_separador(ruta)
    primera = leer_primeras_lineas(ruta, 1)[0]
    sin_cabecera = parece_fecha(primera.split(sep)[0])
    if sin_cabecera:                                       # MetaTrader 4: sin fila de títulos
        df = pd.read_csv(ruta, sep=sep, header=None)
        nombres = {7: ["date", "time", "open", "high", "low", "close", "volume"],
                   6: ["datetime", "open", "high", "low", "close", "volume"],
                   5: ["datetime", "open", "high", "low", "close"]}
        if df.shape[1] in nombres and not (df.shape[1] == 6 and " " not in str(df.iloc[0, 0])):
            df.columns = nombres[df.shape[1]]
        elif df.shape[1] >= 6:
            df = df.iloc[:, :6]
            df.columns = ["date", "time", "open", "high", "low", "close"]
        else:
            raise ErrorAmigable(f"❌ Formato no reconocido ({df.shape[1]} columnas sin cabecera).")
    else:
        df = pd.read_csv(ruta, sep=sep, encoding="utf-8-sig")
        df.columns = [str(c).strip().strip("<>").strip().lower() for c in df.columns]

    renombres = {"o": "open", "h": "high", "l": "low", "c": "close", "apertura": "open",
                 "maximo": "high", "máximo": "high", "minimo": "low", "mínimo": "low", "cierre": "close"}
    df = df.rename(columns=renombres)

    # Columna de fecha/hora
    if "date" in df.columns and "time" in df.columns:
        df["datetime"] = df["date"].astype(str).str.strip() + " " + df["time"].astype(str).str.strip()
        col_fecha = "datetime"
    else:
        candidatas = ["gmt time", "local time", "time (utc)", "datetime", "timestamp", "date", "time", "fecha"]
        col_fecha = next((c for c in candidatas if c in df.columns), None)
    if col_fecha is None:
        raise ErrorAmigable(f"❌ No se encontró columna de fecha. Columnas del archivo: {list(df.columns)}")

    df.index = parsear_fechas(df[col_fecha])
    df.index.name = "fecha"
    if df.index.isna().mean() > 0.5:
        raise ErrorAmigable(
            "❌ No se pudieron leer las fechas del archivo. Primeras líneas:\n   "
            + "\n   ".join(leer_primeras_lineas(ruta, 3))
        )
    df = df[df.index.notna()].sort_index()
    numero = lambda c: pd.to_numeric(df[c], errors="coerce")

    # Volumen: el primero que tenga datos (MT5 suele traer VOL=0 y TICKVOL real)
    col_vol = next((c for c in ("volume", "tickvol", "tick_volume", "tick volume", "vol", "real_volume")
                    if c in df.columns and numero(c).fillna(0).gt(0).any()), None)

    if all(c in df.columns for c in ("open", "high", "low", "close")):
        # --- Archivo de VELAS (OHLC) ---------------------------------------------------
        velas = pd.DataFrame({c: numero(c) for c in ("open", "high", "low", "close")}, index=df.index)
        if col_vol is not None:
            velas["volume"] = numero(col_vol)
        tipo = "velas OHLC"
    else:
        # --- Archivo de TICKS (bid/ask o precio) → se agrupan en velas OHLC --------------
        lado = LADO_PRECIO.lower()
        col_precio = next((c for c in (lado, "bid", "ask", "price", "last", "mid") if c in df.columns), None)
        if col_precio is None:
            raise ErrorAmigable(
                "❌ El archivo no tiene columnas open/high/low/close ni bid/ask/price. "
                f"Columnas encontradas: {list(df.columns)}"
            )
        cols_vol_ticks = [c for c in (f"{lado}volume", f"{lado} volume", "bidvolume", "askvolume") if c in df.columns]
        vol_ticks = numero(col_vol) if col_vol is not None else (
            numero(cols_vol_ticks[0]) if cols_vol_ticks else pd.Series(1.0, index=df.index))  # 1 = contar ticks
        velas = ticks_a_velas(numero(col_precio), vol_ticks)
        tipo = f"ticks ('{col_precio}') agrupados en velas {TEMPORALIDAD}"

    print(f"📄 Formato detectado: separador {sep!r} | fecha '{col_fecha}' | {tipo}"
          + (" | MetaTrader 4 (sin cabecera)" if sin_cabecera else ""))
    return agrupar_en_temporalidad(velas)


def ticks_a_velas(precio, volumen):
    """Convierte ticks en velas OHLC de la TEMPORALIDAD elegida (vectorizado con resample)."""
    regla = TEMPORALIDADES[TEMPORALIDAD][1]
    velas = precio.resample(regla).ohlc()
    velas["volume"] = volumen.resample(regla).sum()
    return velas.dropna(subset=["open"])


def agrupar_en_temporalidad(velas):
    """Si las velas del archivo son más pequeñas que TEMPORALIDAD (ej. M1 → H1), las agrupa."""
    paso = pd.Series(velas.index).diff().median()
    objetivo = pd.Timedelta(TEMPORALIDADES[TEMPORALIDAD][1])
    if pd.notna(paso) and paso < objetivo * 0.99:
        reglas = {"open": "first", "high": "max", "low": "min", "close": "last"}
        if "volume" in velas.columns:
            reglas["volume"] = "sum"
        velas = velas.resample(TEMPORALIDADES[TEMPORALIDAD][1]).agg(reglas).dropna(subset=["open"])
        print(f"🔄 Velas de {paso} agrupadas en velas {TEMPORALIDAD}.")
    elif pd.notna(paso) and paso > objetivo * 1.01:
        print(f"⚠️ El archivo tiene velas de {paso}, más grandes que {TEMPORALIDAD}: se usan tal cual.")
    return velas


def limpiar_ohlc(df):
    """float32, sin huecos ni precios inválidos, orden cronológico, sin duplicados."""
    columnas = ["open", "high", "low", "close"] + (["volume"] if "volume" in df.columns else [])
    df = df[columnas].apply(pd.to_numeric, errors="coerce").astype("float32")   # float32 = mitad de RAM
    filas_iniciales = len(df)
    df = df[df.index.notna()].dropna(subset=["open", "high", "low", "close"])
    df = df[(df[["open", "high", "low", "close"]] > 0).all(axis=1)]
    df = df.sort_index()                                   # Cronológico: antiguo → reciente
    df = df[~df.index.duplicated(keep="last")]
    if ELIMINAR_VELAS_SIN_VOLUMEN and "volume" in df.columns:
        con_volumen = df["volume"] > 0
        if con_volumen.mean() >= 0.5:                      # Solo si no vacía el archivo
            df = df[con_volumen]
    descartadas = filas_iniciales - len(df)
    if descartadas:
        print(f"🧹 Se descartaron {descartadas:,} velas inválidas, duplicadas o sin volumen.")
    df.index.name = "fecha"
    return df


# ------------------------------------------------------------------------------
# A) DESCARGA INCREMENTAL DESDE DUKASCOPY CON CACHÉ EN GOOGLE DRIVE
# ------------------------------------------------------------------------------
#   • 1ª vez: descarga desde FECHA_INICIO_DESCARGA hasta hoy y lo guarda en Drive (Parquet).
#   • Siguientes veces: lee el archivo de Drive y SOLO descarga desde la última vela guardada.
#   • Guarda tras cada bloque: si se corta internet, al volver a ejecutar continúa donde iba.
BLOQUE_DESCARGA = {"M1": pd.Timedelta(days=20), "M5": pd.Timedelta(days=100), "M10": pd.Timedelta(days=200),
                   "M15": pd.Timedelta(days=300), "M30": pd.Timedelta(days=600)}  # ≤ 30.000 velas por petición


def ruta_cache():
    limpio = re.sub(r"[^A-Za-z0-9]+", "", INSTRUMENTO.upper())
    return os.path.join(RUTA_DATOS, f"{limpio}_{TEMPORALIDAD}_{LADO_PRECIO}.parquet")


def leer_inicio_solicitado(ruta):
    """Fecha más antigua ya pedida a Dukascopy (para no volver a pedir tramos sin datos)."""
    try:
        with open(ruta + ".json") as f:
            return pd.Timestamp(json.load(f)["inicio_solicitado"], tz=timezone.utc)
    except Exception:
        return None


def guardar_inicio_solicitado(ruta, inicio):
    with open(ruta + ".json", "w") as f:
        json.dump({"inicio_solicitado": inicio.tz_convert(None).isoformat()}, f)


def guardar_parquet(df, ruta):
    temporal = ruta + ".tmp"
    df.to_parquet(temporal)
    os.replace(temporal, ruta)                             # escritura atómica: nunca queda a medias


def validar_instrumento():
    import dukascopy_python.instruments as ins
    import difflib
    validos = {v for k, v in vars(ins).items() if k.startswith("INSTRUMENT_") and isinstance(v, str)}
    if INSTRUMENTO not in validos:
        parecidos = difflib.get_close_matches(INSTRUMENTO, sorted(validos), n=8, cutoff=0.4)
        raise ErrorAmigable(
            f"❌ '{INSTRUMENTO}' no es un instrumento de Dukascopy.\n"
            f"   Quizá quisiste decir: {parecidos or 'EUR/USD, XAU/USD, BTC/USD, E_NQ-100'}\n"
            "   Cambia INSTRUMENTO en la Celda 2."
        )


def descargar_tramo(inicio, fin):
    """Descarga velas [inicio, fin] (UTC) con la API pública de Dukascopy."""
    import dukascopy_python
    logging.getLogger("DUKASCRIPT").propagate = False      # silencia los mensajes internos
    lado = dukascopy_python.OFFER_SIDE_BID if LADO_PRECIO == "BID" else dukascopy_python.OFFER_SIDE_ASK
    datos = dukascopy_python.fetch(INSTRUMENTO, TEMPORALIDADES[TEMPORALIDAD][0], lado,
                                   inicio.to_pydatetime(), fin.to_pydatetime(), max_retries=7)
    if datos is None or len(datos) == 0:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"], dtype="float32")
    datos.index = pd.DatetimeIndex(datos.index).tz_convert(None)   # UTC sin zona horaria
    datos.columns = [str(c).lower() for c in datos.columns]
    return datos[["open", "high", "low", "close", "volume"]].astype("float32")


def descargar_o_actualizar():
    validar_instrumento()
    ruta = ruta_cache()
    ahora = pd.Timestamp.now(tz=timezone.utc).floor("min")
    inicio_deseado = pd.Timestamp(FECHA_INICIO_DESCARGA, tz=timezone.utc)

    if os.path.isfile(ruta):
        datos = pd.read_parquet(ruta)
        print(f"💾 Datos en Drive: {len(datos):,} velas ({datos.index[0]} → {datos.index[-1]})")
    else:
        datos = pd.DataFrame(columns=["open", "high", "low", "close", "volume"], dtype="float32")
        print(f"🆕 No hay datos guardados de {INSTRUMENTO} {TEMPORALIDAD}: primera descarga completa.")

    tramos = []
    if datos.empty:
        tramos.append((inicio_deseado, ahora))
    else:
        primera = datos.index[0].tz_localize(timezone.utc)
        ultima = datos.index[-1].tz_localize(timezone.utc)
        ya_pedido = leer_inicio_solicitado(ruta) or primera
        if inicio_deseado < min(primera, ya_pedido) - pd.Timedelta(days=3):  # piden más historia
            tramos.append((inicio_deseado, min(primera, ya_pedido)))
        tramos.append((ultima, ahora))                      # actualización (re-baja la última vela)

    bloque = BLOQUE_DESCARGA.get(TEMPORALIDAD, pd.Timedelta(days=3650))
    nuevas_total = 0
    t0 = time.time()
    for inicio, fin in tramos:
        cursor = inicio
        while cursor < fin:
            hasta = min(cursor + bloque, fin)
            try:
                nuevas = descargar_tramo(cursor, hasta)
            except Exception as error:
                if not datos.empty:
                    guardar_parquet(datos, ruta)
                raise ErrorAmigable(
                    f"❌ Falló la descarga ({cursor:%Y-%m-%d} → {hasta:%Y-%m-%d}): {error}\n"
                    "   Lo ya descargado está guardado en Drive. Vuelve a ejecutar esta celda para continuar."
                ) from None
            if len(nuevas):
                antes = len(datos)
                datos = pd.concat([datos, nuevas]) if not datos.empty else nuevas
                datos = datos[~datos.index.duplicated(keep="last")].sort_index()
                nuevas_total += len(datos) - antes
                guardar_parquet(datos, ruta)
            print(f"   ⬇️ {cursor:%Y-%m-%d} → {hasta:%Y-%m-%d}: {len(nuevas):,} velas recibidas "
                  f"| total {len(datos):,} | {time.time() - t0:,.0f} s", flush=True)
            cursor = hasta

    if tramos[0][0] == inicio_deseado:   # el tramo antiguo se completó: no volver a pedirlo
        guardar_inicio_solicitado(ruta, inicio_deseado)
    if datos.empty:
        raise ErrorAmigable(f"❌ Dukascopy no devolvió datos para {INSTRUMENTO} desde {FECHA_INICIO_DESCARGA}. "
                            "Prueba otra fecha de inicio u otro instrumento.")
    print(f"✅ Datos actualizados: {nuevas_total:,} velas nuevas | guardado en {ruta}")
    datos = datos[datos.index >= inicio_deseado.tz_localize(None)]
    # La última vela puede estar aún formándose: no se usa en el backtest (se completará
    # en la próxima actualización).
    duracion = pd.Timedelta(TEMPORALIDADES[TEMPORALIDAD][1])
    if len(datos) and datos.index[-1] + duracion > ahora.tz_localize(None):
        datos = datos.iloc[:-1]
    return datos


df_is = df_oos = df_oos_completo = None    # evita usar datos de una ejecución anterior si algo falla
try:
    if FUENTE_DATOS == "DESCARGAR":
        df_total = limpiar_ohlc(descargar_o_actualizar())
        NOMBRE_ACTIVO = f"{re.sub(r'[^A-Za-z0-9]+', '', INSTRUMENTO.upper())}_{TEMPORALIDAD}"
    else:
        df_total = limpiar_ohlc(cargar_csv_ohlc(os.path.join(RUTA_ACTIVOS, NOMBRE_ARCHIVO)))
        NOMBRE_ACTIVO = f"{os.path.splitext(NOMBRE_ARCHIVO)[0]}_{TEMPORALIDAD}"

    # Periodo del indicador más largo → número de velas a purgar al inicio del OOS
    PERIODO_PURGA = int(max(np.max(GRAN_MALLE_PARAMETROS["ma_lenta"]),
                            np.max(GRAN_MALLE_PARAMETROS["ma_rapida"]),
                            np.max(GRAN_MALLE_PARAMETROS["rsi_periodo"]) + 1))

    n_total = len(df_total)
    n_is = int(n_total * PORCENTAJE_IS)
    if n_is < PERIODO_PURGA * 3 or (n_total - n_is) < PERIODO_PURGA * 2:
        raise ErrorAmigable(
            f"❌ Muy pocos datos ({n_total:,} velas) para indicadores de hasta {PERIODO_PURGA} periodos. "
            "Usa un histórico más largo o reduce los periodos máximos de la malla."
        )

    # --- División estricta: el IS nunca ve datos del OOS -------------------------------
    df_is = df_total.iloc[:n_is].copy()
    # El OOS completo empieza justo después del IS. Sus indicadores se calculan SOLO con
    # datos OOS (nunca con el final del IS), así que sus primeras PERIODO_PURGA velas son
    # el "calentamiento" de los indicadores.
    df_oos_completo = df_total.iloc[n_is:].copy()
    # --- PURGA: eliminamos esas velas iniciales afectadas por el indicador más largo -----
    #     No se opera ni se mide nada en ellas → cero contaminación entre IS y OOS.
    df_oos = df_oos_completo.iloc[PERIODO_PURGA:].copy()

    # Frecuencia de las velas y nº de velas por año (para anualizar el Sharpe)
    FRECUENCIA = pd.Series(df_total.index).diff().median()
    anios = (df_total.index[-1] - df_total.index[0]).total_seconds() / (365.25 * 24 * 3600)
    BARRAS_POR_ANIO = float(n_total / anios) if anios > 0 else 252.0

    del df_total
    gc.collect()

    resumen = pd.DataFrame({
        "Velas": [len(df_is), PERIODO_PURGA, len(df_oos)],
        "Desde": [df_is.index[0], df_oos_completo.index[0], df_oos.index[0]],
        "Hasta": [df_is.index[-1], df_oos_completo.index[PERIODO_PURGA - 1], df_oos.index[-1]],
    }, index=["In-Sample (entrenamiento)", "Purga (calentamiento OOS)", "Out-Of-Sample (validación)"])
    print(f"✅ '{NOMBRE_ACTIVO}' cargado: {n_total:,} velas | frecuencia {FRECUENCIA} | "
          f"~{BARRAS_POR_ANIO:,.0f} velas/año | RAM usada: "
          f"{(df_is.memory_usage().sum() + df_oos_completo.memory_usage().sum()) / 1024**2:,.1f} MB")
    display(resumen)
except ErrorAmigable:
    raise
except Exception as error:
    raise ErrorAmigable(f"❌ Error inesperado obteniendo los datos: {error}") from None

## 4️⃣ Motor de generación masiva acelerado por hardware (GPU)

**Estrategia:** cruce de medias móviles simples (rápida / lenta) con filtro RSI.

| Dirección | Entrada | Salida |
|---|---|---|
| `COMPRAS_SOLO` | MA rápida cruza **arriba** de la lenta y RSI < umbral | MA rápida cruza abajo |
| `VENTAS_SOLO` | MA rápida cruza **abajo** de la lenta y RSI > 100 − umbral | MA rápida cruza arriba |

**Cómo se usa la GPU:**
- **CuPy** calcula todas las SMA y RSI de la malla directamente en la VRAM.
- Un **kernel CUDA** (C++ compilado al vuelo por CuPy) hace que **cada hilo de la GPU simula una estrategia completa**. Las señales se generan dentro del kernel, sin crear matrices gigantes en memoria.
- **Filtro de dirección estricto:** en `COMPRAS_SOLO` las entradas cortas (`short_entries`) son `False` absoluto; en `VENTAS_SOLO` lo son las entradas largas (`entries`).
- Las órdenes se ejecutan en la **apertura de la vela siguiente** a la señal, para no mirar el futuro.

**¿Por qué no `Portfolio.from_signals()` para toda la malla?** La versión gratuita de VectorBT simula en **CPU** (Numba) y necesitaría matrices de señales de decenas de GB para 100.000+ estrategias. Por eso la malla completa se simula en el kernel CUDA y VectorBT `from_signals()` se usa en la Celda 5 para las mejores estrategias. Al final de esta celda se comprueba que ambos motores dan **los mismos resultados**.

In [ ]:
# ==============================================================================
# CELDA 4 · MOTOR DE GENERACIÓN MASIVA ACELERADO POR GPU (CuPy + kernel CUDA)
# ==============================================================================
# Arquitectura (por qué está hecho así):
#   • VectorBT (versión open-source) simula carteras con Numba en CPU; no tiene backend
#     GPU y `vbt.Param` solo existe en VectorBT PRO. Meter 100.000+ columnas de señales en
#     `Portfolio.from_signals()` exigiría decenas de GB de matrices booleanas en RAM.
#   • Solución HPC: un KERNEL CUDA (C++ compilado por CuPy) en el que cada hilo de la GPU simula UNA combinación de
#     parámetros sobre todas las velas (la RTX Pro 6000 ejecuta decenas de miles de hilos a
#     la vez). Las señales se calculan dentro del kernel → no se crean matrices gigantes.
#   • Los indicadores (SMA y RSI) se calculan una sola vez por periodo con CuPy en la GPU
#     mediante sumas acumuladas (100% vectorizado, sin bucles de Python).
#   • Después, VectorBT `Portfolio.from_signals()` vuelve a simular las mejores
#     estrategias (Celda 5) y aquí verificamos que el motor GPU coincide con VectorBT.
#   • Si no hay GPU, la misma lógica (versión Python) se compila con Numba en CPU paralelo.
# ==============================================================================
import gc
import math
import time

import numpy as np
import pandas as pd
import vectorbt as vbt
from numba import njit, prange

try:
    import cupy as cp
except ImportError:
    cp = None

xp = cp if GPU_DISPONIBLE else np          # xp = "NumPy de la GPU" o NumPy normal
MOTOR = "GPU (kernel CUDA)" if KERNEL_CUDA_OK else "CPU paralelo (Numba)"
ES_COMPRA = DIRECCION_OPERACION == "COMPRAS_SOLO"
COLUMNAS_METRICAS = ["retorno_total", "profit_factor", "trades", "max_drawdown", "win_rate", "sharpe"]

if GPU_DISPONIBLE:
    # Límite de VRAM para CuPy: nunca más de FRACCION_MAX_VRAM de la memoria de la GPU
    cp.get_default_memory_pool().set_limit(fraction=FRACCION_MAX_VRAM)


def a_numpy(arr):
    """Copia un arreglo de la GPU a la RAM (si ya está en RAM no hace nada)."""
    return cp.asnumpy(arr) if (cp is not None and isinstance(arr, cp.ndarray)) else np.asarray(arr)


# ------------------------------------------------------------------------------
# 1) Indicadores vectorizados para TODOS los periodos a la vez (en la GPU)
# ------------------------------------------------------------------------------
def calcular_smas(close, periodos):
    """Matriz (n_periodos, n_velas) de medias móviles simples. NaN durante el calentamiento.
    Cada fila se calcula con una suma acumulada en la GPU: O(n) por periodo, sin bucles por vela."""
    c = xp.asarray(close, dtype=xp.float64)
    acumulada = xp.concatenate([xp.zeros(1, dtype=xp.float64), xp.cumsum(c)])
    salida = xp.full((len(periodos), c.shape[0]), xp.nan, dtype=xp.float32)
    for i, p in enumerate(periodos):
        p = int(p)
        salida[i, p - 1:] = ((acumulada[p:] - acumulada[:-p]) / p).astype(xp.float32)
    return salida


def calcular_rsis(close, periodos):
    """RSI con medias simples de ganancias/pérdidas (igual que vbt.RSI por defecto)."""
    c = xp.asarray(close, dtype=xp.float64)
    delta = xp.diff(c)
    acum_sube = xp.concatenate([xp.zeros(1), xp.cumsum(xp.maximum(delta, 0.0))])
    acum_baja = xp.concatenate([xp.zeros(1), xp.cumsum(xp.maximum(-delta, 0.0))])
    salida = xp.full((len(periodos), c.shape[0]), xp.nan, dtype=xp.float32)
    for i, p in enumerate(periodos):
        p = int(p)
        sube = acum_sube[p:] - acum_sube[:-p]       # suma de subidas de las últimas p velas
        baja = acum_baja[p:] - acum_baja[:-p]
        total = sube + baja
        rsi = 100.0 * sube / xp.where(total > 0, total, xp.nan)
        salida[i, p:] = rsi.astype(xp.float32)
    return salida


def indicadores_desde_precios(open_, close):
    """SMA y RSI de todos los periodos de la malla a partir de arreglos de precios (GPU)."""
    close = xp.asarray(close, dtype=xp.float32)
    return {
        "open": xp.asarray(open_, dtype=xp.float32),
        "close": close,
        "ma": calcular_smas(close, PERIODOS_MA),
        "rsi": calcular_rsis(close, PERIODOS_RSI),
    }


def construir_indicadores(df):
    """Calcula en la GPU todas las SMA y RSI necesarias para la malla, sobre `df`."""
    return indicadores_desde_precios(df["open"].to_numpy(np.float32), df["close"].to_numpy(np.float32))


# ------------------------------------------------------------------------------
# 2) Malla de parámetros (equivalente a vbt.Param de VectorBT PRO, con producto cartesiano)
# ------------------------------------------------------------------------------
PERIODOS_MA = np.unique(np.concatenate([np.asarray(GRAN_MALLE_PARAMETROS["ma_rapida"]),
                                        np.asarray(GRAN_MALLE_PARAMETROS["ma_lenta"])])).astype(np.int64)
PERIODOS_RSI = np.unique(np.asarray(GRAN_MALLE_PARAMETROS["rsi_periodo"])).astype(np.int64)


def construir_malla():
    rap = np.unique(GRAN_MALLE_PARAMETROS["ma_rapida"])          # np.unique: sin valores repetidos
    len_ = np.unique(GRAN_MALLE_PARAMETROS["ma_lenta"])
    rsi_p = np.unique(GRAN_MALLE_PARAMETROS["rsi_periodo"])
    umb = np.unique(np.asarray(GRAN_MALLE_PARAMETROS["rsi_umbral"], dtype=np.float32))
    # indexing="ij": el umbral varía más rápido → hilos vecinos de la GPU leen las mismas
    # medias móviles (accesos a memoria agrupados = más velocidad).
    R, L, P, U = np.meshgrid(rap, len_, rsi_p, umb, indexing="ij")
    R, L, P, U = R.ravel(), L.ravel(), P.ravel(), U.ravel()
    validas = R < L
    malla = pd.DataFrame({
        "ma_rapida": R[validas].astype(np.int32),
        "ma_lenta": L[validas].astype(np.int32),
        "rsi_periodo": P[validas].astype(np.int32),
        "rsi_umbral": U[validas].astype(np.float32),
    })
    malla["idx_rapida"] = np.searchsorted(PERIODOS_MA, malla["ma_rapida"]).astype(np.int32)
    malla["idx_lenta"] = np.searchsorted(PERIODOS_MA, malla["ma_lenta"]).astype(np.int32)
    malla["idx_rsi"] = np.searchsorted(PERIODOS_RSI, malla["rsi_periodo"]).astype(np.int32)
    return malla


# ------------------------------------------------------------------------------
# 3) Simulación de UNA combinación (se compila para GPU y para CPU con el mismo código)
# ------------------------------------------------------------------------------
#   Reglas de ejecución (idénticas a la validación con VectorBT):
#   • La señal se calcula con el CIERRE de la vela t-1 y se ejecuta en la APERTURA de la
#     vela t (sin mirar el futuro). La cartera se valora al cierre de cada vela.
#   • Se invierte todo el capital (size=inf) pagando COMISION_PCT en la entrada y la salida.
def _simular_combinacion(k, open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                         es_compra, comision, inicio, fin, retraso, factor_anual, out):
    f = c_rap[k]
    s = c_len[k]
    r = c_rsi[k]
    umbral_compra = c_umb[k]
    umbral_venta = 100.0 - umbral_compra
    sin_filtro_rsi = umbral_compra >= 100.0      # umbral 100 = sin filtro (aunque el RSI sea 100 o NaN)

    capital = 1.0            # capital realizado (normalizado a 1)
    valor_prev = 1.0         # valor de la cartera al cierre anterior
    pico = 1.0
    max_dd = 0.0
    en_posicion = False
    unidades = 0.0
    precio_entrada = 0.0
    capital_entrada = 0.0
    n_trades = 0
    n_ganadores = 0
    ganancias = 0.0
    perdidas = 0.0
    suma_r = 0.0
    suma_r2 = 0.0
    n_r = 0

    for t in range(inicio, fin):
        j = t - retraso                              # vela de la señal (retraso=1: la anterior)
        mf = ma[f, j]
        ms = ma[s, j]
        mf_prev = ma[f, j - 1]
        ms_prev = ma[s, j - 1]
        cruce_arriba = (mf > ms) and (mf_prev <= ms_prev)   # NaN → False automáticamente
        cruce_abajo = (mf < ms) and (mf_prev >= ms_prev)
        valor_rsi = rsi[r, j]
        if es_compra:
            senal_entrada = cruce_arriba and (sin_filtro_rsi or valor_rsi < umbral_compra)
            senal_salida = cruce_abajo
        else:
            senal_entrada = cruce_abajo and (sin_filtro_rsi or valor_rsi > umbral_venta)
            senal_salida = cruce_arriba

        precio = open_[t]
        if en_posicion and senal_salida:
            if es_compra:
                capital = unidades * precio * (1.0 - comision)
            else:
                capital = capital_entrada + unidades * (precio_entrada * (1.0 - comision)
                                                        - precio * (1.0 + comision))
            pnl = capital - capital_entrada
            n_trades += 1
            if pnl > 0:
                n_ganadores += 1
                ganancias += pnl
            else:
                perdidas -= pnl
            en_posicion = False
        elif (not en_posicion) and senal_entrada and capital > 0:
            capital_entrada = capital
            precio_entrada = precio
            unidades = capital / (precio * (1.0 + comision))
            en_posicion = True

        # Valoración al cierre de la vela (mark-to-market)
        if en_posicion:
            if es_compra:
                valor = unidades * close[t]
            else:
                valor = capital_entrada + unidades * (precio_entrada * (1.0 - comision) - close[t])
        else:
            valor = capital

        if valor_prev > 0:
            ret = valor / valor_prev - 1.0
            suma_r += ret
            suma_r2 += ret * ret
            n_r += 1
        valor_prev = valor
        if valor > pico:
            pico = valor
        if pico > 0:
            dd = (pico - valor) / pico
            if dd > max_dd:
                max_dd = dd

    out[k, 0] = valor_prev - 1.0
    if perdidas > 0:
        out[k, 1] = ganancias / perdidas
    elif ganancias > 0:
        out[k, 1] = math.inf
    else:
        out[k, 1] = 0.0
    out[k, 2] = n_trades
    out[k, 3] = max_dd
    out[k, 4] = n_ganadores / n_trades if n_trades > 0 else math.nan
    if n_r > 1:
        media = suma_r / n_r
        var = (suma_r2 - suma_r * suma_r / n_r) / (n_r - 1)
        out[k, 5] = media / math.sqrt(var) * factor_anual if var > 1e-30 else math.nan
    else:
        out[k, 5] = math.nan


# --- Versión CPU (Numba paralelo: un núcleo por bloque de combinaciones) --------------
_simular_cpu = njit(cache=False)(_simular_combinacion)


@njit(parallel=True, cache=False)
def _backtest_cpu(open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                  es_compra, comision, inicio, fin, retraso, factor_anual, out):
    for k in prange(out.shape[0]):
        _simular_cpu(k, open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                     es_compra, comision, inicio, fin, retraso, factor_anual, out)


# --- Versión GPU: kernel CUDA C compilado por CuPy (NVRTC). Un hilo = una combinación ---
#     Es la traducción línea a línea de _simular_combinacion (arriba). Se usa CuPy en lugar
#     de Numba CUDA porque CuPy ya funciona en la GPU de Colab sin paquetes adicionales.
CODIGO_CUDA = r"""
#define INF_D __longlong_as_double(0x7ff0000000000000LL)
#define NAN_D __longlong_as_double(0x7ff8000000000000LL)

extern "C" __global__ void backtest_kernel(
    const float* __restrict__ open_, const float* __restrict__ close,
    const float* __restrict__ ma, const float* __restrict__ rsi,
    const int* __restrict__ c_rap, const int* __restrict__ c_len,
    const int* __restrict__ c_rsi, const float* __restrict__ c_umb,
    const int es_compra, const double comision, const long long inicio,
    const long long fin, const long long n, const long long retraso,
    const double factor_anual, const long long n_comb, double* __restrict__ out)
{
    const long long k = (long long)blockDim.x * blockIdx.x + threadIdx.x;
    if (k >= n_comb) return;

    const float* ma_f = ma + (long long)c_rap[k] * n;   // fila de la MA rápida
    const float* ma_s = ma + (long long)c_len[k] * n;   // fila de la MA lenta
    const float* rsi_r = rsi + (long long)c_rsi[k] * n; // fila del RSI
    const float umbral_compra = c_umb[k];
    const float umbral_venta = 100.0f - umbral_compra;
    const bool sin_filtro_rsi = umbral_compra >= 100.0f;   // umbral 100 = sin filtro RSI

    double capital = 1.0, valor_prev = 1.0, pico = 1.0, max_dd = 0.0;
    bool en_posicion = false;
    double unidades = 0.0, precio_entrada = 0.0, capital_entrada = 0.0;
    long long n_trades = 0, n_ganadores = 0, n_r = 0;
    double ganancias = 0.0, perdidas = 0.0, suma_r = 0.0, suma_r2 = 0.0;

    for (long long t = inicio; t < fin; ++t) {
        const long long j = t - retraso;               // vela de la señal
        const float mf = ma_f[j], ms = ma_s[j], mf_prev = ma_f[j - 1], ms_prev = ma_s[j - 1];
        const bool cruce_arriba = (mf > ms) && (mf_prev <= ms_prev);   // NaN -> false
        const bool cruce_abajo = (mf < ms) && (mf_prev >= ms_prev);
        const float valor_rsi = rsi_r[j];
        bool senal_entrada, senal_salida;
        if (es_compra) {
            senal_entrada = cruce_arriba && (sin_filtro_rsi || valor_rsi < umbral_compra);
            senal_salida = cruce_abajo;
        } else {
            senal_entrada = cruce_abajo && (sin_filtro_rsi || valor_rsi > umbral_venta);
            senal_salida = cruce_arriba;
        }

        const double precio = (double)open_[t];
        if (en_posicion && senal_salida) {
            if (es_compra) {
                capital = unidades * precio * (1.0 - comision);
            } else {
                capital = capital_entrada + unidades * (precio_entrada * (1.0 - comision)
                                                        - precio * (1.0 + comision));
            }
            const double pnl = capital - capital_entrada;
            n_trades += 1;
            if (pnl > 0) { n_ganadores += 1; ganancias += pnl; }
            else { perdidas -= pnl; }
            en_posicion = false;
        } else if (!en_posicion && senal_entrada && capital > 0) {
            capital_entrada = capital;
            precio_entrada = precio;
            unidades = capital / (precio * (1.0 + comision));
            en_posicion = true;
        }

        double valor;
        if (en_posicion) {
            const double c = (double)close[t];
            if (es_compra) valor = unidades * c;
            else valor = capital_entrada + unidades * (precio_entrada * (1.0 - comision) - c);
        } else {
            valor = capital;
        }

        if (valor_prev > 0) {
            const double ret = valor / valor_prev - 1.0;
            suma_r += ret;
            suma_r2 += ret * ret;
            n_r += 1;
        }
        valor_prev = valor;
        if (valor > pico) pico = valor;
        if (pico > 0) {
            const double dd = (pico - valor) / pico;
            if (dd > max_dd) max_dd = dd;
        }
    }

    double* o = out + k * 6;
    o[0] = valor_prev - 1.0;
    if (perdidas > 0) o[1] = ganancias / perdidas;
    else if (ganancias > 0) o[1] = INF_D;
    else o[1] = 0.0;
    o[2] = (double)n_trades;
    o[3] = max_dd;
    o[4] = n_trades > 0 ? (double)n_ganadores / (double)n_trades : NAN_D;
    if (n_r > 1) {
        const double media = suma_r / n_r;
        const double var = (suma_r2 - suma_r * suma_r / n_r) / (n_r - 1);
        o[5] = var > 1e-30 ? media / sqrt(var) * factor_anual : NAN_D;
    } else {
        o[5] = NAN_D;
    }
}
"""

if KERNEL_CUDA_OK:
    _backtest_gpu = cp.RawKernel(CODIGO_CUDA, "backtest_kernel")


def backtest_masivo(ind, malla, inicio, fin=None, retraso=1, comision=None, mostrar=False):
    """Simula todas las filas de `malla` entre las velas [inicio, fin) y devuelve sus métricas.
    retraso = velas entre la señal y la orden (1 = apertura siguiente). comision: por defecto COMISION_PCT."""
    n_comb = len(malla)
    retraso = max(int(retraso), 1)
    inicio = max(int(inicio), retraso + 1)
    n_total = int(ind["close"].shape[0])
    fin = n_total if fin is None else min(int(fin), n_total)
    comision = COMISION_PCT if comision is None else comision
    factor_anual = math.sqrt(BARRAS_POR_ANIO)
    columnas = [malla[c].to_numpy() for c in ("idx_rapida", "idx_lenta", "idx_rsi", "rsi_umbral")]

    if KERNEL_CUDA_OK:
        # Tamaño de lote según la VRAM libre (16 B de parámetros + 48 B de métricas por combinación)
        libre, _ = cp.cuda.runtime.memGetInfo()
        lote = int(min(LOTE_MAX_COMBINACIONES, max(1, libre * FRACCION_MAX_VRAM // 64)))
        resultados = np.empty((n_comb, len(COLUMNAS_METRICAS)), dtype=np.float64)
        arrays = [cp.ascontiguousarray(ind[c]) for c in ("open", "close", "ma", "rsi")]
        hilos = 128
        for a in range(0, n_comb, lote):
            b = min(a + lote, n_comb)
            params_gpu = [cp.asarray(columnas[0][a:b], dtype=cp.int32), cp.asarray(columnas[1][a:b], dtype=cp.int32),
                          cp.asarray(columnas[2][a:b], dtype=cp.int32), cp.asarray(columnas[3][a:b], dtype=cp.float32)]
            out_gpu = cp.empty((b - a, len(COLUMNAS_METRICAS)), dtype=cp.float64)
            bloques = (b - a + hilos - 1) // hilos
            _backtest_gpu((bloques,), (hilos,), (*arrays, *params_gpu, np.int32(ES_COMPRA),
                                                 np.float64(comision), np.int64(inicio), np.int64(fin),
                                                 np.int64(n_total), np.int64(retraso),
                                                 np.float64(factor_anual), np.int64(b - a), out_gpu))
            cp.cuda.Stream.null.synchronize()
            resultados[a:b] = cp.asnumpy(out_gpu)
            del params_gpu, out_gpu
            if mostrar:
                print(f"   ⚡ Lote {a:,}–{b:,} de {n_comb:,} completado")
        del arrays
    else:
        resultados = np.empty((n_comb, len(COLUMNAS_METRICAS)), dtype=np.float64)
        _backtest_cpu(a_numpy(ind["open"]), a_numpy(ind["close"]), a_numpy(ind["ma"]),
                      a_numpy(ind["rsi"]), *columnas, ES_COMPRA, comision, inicio, fin, retraso,
                      factor_anual, resultados)

    met = pd.DataFrame(resultados, columns=COLUMNAS_METRICAS, index=malla.index)
    met["trades"] = met["trades"].astype(np.int64)
    met["calmar"] = met["retorno_total"] / met["max_drawdown"].replace(0, np.nan)
    return met


# ------------------------------------------------------------------------------
# 4) Simulación oficial con VectorBT Portfolio.from_signals (para Top y validación)
# ------------------------------------------------------------------------------
def _desplazar(m, retraso=1):
    """Retrasa las señales: señal al cierre de t-retraso → orden en la apertura de t."""
    out = np.zeros_like(m)
    out[:, retraso:] = m[:, :-retraso]
    return out


def generar_senales(ind_np, combos, retraso=1):
    """Matrices de señales (velas × combinaciones) totalmente vectorizadas con NumPy."""
    ma, rsi = ind_np["ma"], ind_np["rsi"]
    f = ma[combos["idx_rapida"].to_numpy()]
    s = ma[combos["idx_lenta"].to_numpy()]
    r = rsi[combos["idx_rsi"].to_numpy()]
    u = combos["rsi_umbral"].to_numpy(np.float32)[:, None]
    f_prev = np.full_like(f, np.nan)
    f_prev[:, 1:] = f[:, :-1]
    s_prev = np.full_like(s, np.nan)
    s_prev[:, 1:] = s[:, :-1]
    with np.errstate(invalid="ignore"):
        cruce_arriba = (f > s) & (f_prev <= s_prev)
        cruce_abajo = (f < s) & (f_prev >= s_prev)
        sin_filtro = u >= 100.0                   # umbral 100 = sin filtro RSI
        if ES_COMPRA:
            entradas, salidas = cruce_arriba & (sin_filtro | (r < u)), cruce_abajo
        else:
            entradas, salidas = cruce_abajo & (sin_filtro | (r > 100.0 - u)), cruce_arriba
    return _desplazar(entradas, retraso).T, _desplazar(salidas, retraso).T


def simular_vectorbt(df, ind_np, combos, inicio=0, fin=None, retraso=1, comision=None):
    """Simula `combos` con VectorBT entre las velas [inicio, fin) y devuelve (métricas, portfolio)."""
    retraso = max(int(retraso), 1)
    inicio = max(int(inicio), retraso + 1)
    fin = len(df) if fin is None else min(int(fin), len(df))
    entradas, salidas = generar_senales(ind_np, combos, retraso)
    entradas, salidas = entradas[inicio:fin], salidas[inicio:fin]
    vacias = np.zeros_like(entradas)            # Matriz de False absolutos
    indice = df.index[inicio:fin]
    columnas = pd.RangeIndex(len(combos))
    a_df = lambda m: pd.DataFrame(m, index=indice, columns=columnas)

    if ES_COMPRA:   # COMPRAS_SOLO → short_entries forzadas a False absoluto
        largas_e, largas_s, cortas_e, cortas_s = entradas, salidas, vacias, vacias
    else:           # VENTAS_SOLO → entries (largas) forzadas a False absoluto
        largas_e, largas_s, cortas_e, cortas_s = vacias, vacias, entradas, salidas

    pf = vbt.Portfolio.from_signals(
        close=df["close"].iloc[inicio:fin],
        price=df["open"].iloc[inicio:fin],       # Ejecución en la apertura de la vela
        entries=a_df(largas_e), exits=a_df(largas_s),
        short_entries=a_df(cortas_e), short_exits=a_df(cortas_s),
        size=np.inf, fees=COMISION_PCT if comision is None else comision,
        init_cash=CAPITAL_INICIAL, freq=FRECUENCIA,
    )
    rets = pf.returns()
    cerrados = pf.trades.closed
    met = pd.DataFrame({
        "retorno_total": pf.total_return().to_numpy(),
        "profit_factor": cerrados.profit_factor().to_numpy(),
        "trades": cerrados.count().to_numpy(),
        "max_drawdown": np.abs(pf.max_drawdown().to_numpy()),
        "win_rate": cerrados.win_rate().to_numpy(),
        "sharpe": (rets.mean() / rets.std(ddof=1) * np.sqrt(BARRAS_POR_ANIO)).to_numpy(),
    }, index=combos.index)
    met["calmar"] = met["retorno_total"] / met["max_drawdown"].replace(0, np.nan)
    return met, pf


# ==============================================================================
# EJECUCIÓN: malla completa sobre el bloque In-Sample
# ==============================================================================
resultados_is = None    # evita usar resultados de una ejecución anterior si algo falla
if globals().get("df_is") is None:
    raise ErrorAmigable("❌ No hay datos cargados: ejecuta primero la Celda 3 sin errores.")
try:
    print(f"🚀 Motor de simulación: {MOTOR} | Indicadores: {'CuPy (GPU)' if GPU_DISPONIBLE else 'NumPy (CPU)'}")
    t0 = time.time()
    malla = construir_malla()
    ind_is = construir_indicadores(df_is)          # Indicadores SOLO con datos In-Sample
    if GPU_DISPONIBLE:
        cp.cuda.Stream.null.synchronize()
    print(f"   📐 {len(PERIODOS_MA)} SMAs + {len(PERIODOS_RSI)} RSIs calculadas en {time.time() - t0:.2f} s "
          f"({ind_is['ma'].nbytes / 1024**2:,.1f} MB en {'VRAM' if GPU_DISPONIBLE else 'RAM'})")

    t1 = time.time()
    metricas_is = backtest_masivo(ind_is, malla, inicio=2, mostrar=True)
    dt = time.time() - t1
    resultados_is = pd.concat([malla[["ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"]], metricas_is], axis=1)
    velas_simuladas = len(malla) * len(df_is)
    print(f"✅ {len(malla):,} estrategias simuladas sobre {len(df_is):,} velas en {dt:,.2f} s "
          f"→ {velas_simuladas / max(dt, 1e-9) / 1e9:,.2f} mil millones de velas·estrategia/seg")

    # --- Verificación de coherencia: motor GPU vs VectorBT en 5 combinaciones al azar ----
    activas = resultados_is[resultados_is["trades"] > 0]
    muestra = (activas if len(activas) else resultados_is).sample(
        min(5, len(activas) or len(resultados_is)), random_state=42).index
    ind_is_np = {k: a_numpy(v) for k, v in ind_is.items()}
    met_vbt, _ = simular_vectorbt(df_is, ind_is_np, malla.loc[muestra], inicio=2)
    diferencia = (met_vbt["retorno_total"] - metricas_is.loc[muestra, "retorno_total"]).abs().max()
    trades_ok = (met_vbt["trades"].to_numpy() == metricas_is.loc[muestra, "trades"].to_numpy()).all()
    estado = "✅ coincide" if (diferencia < 1e-3 and trades_ok) else "⚠️ revisar"
    print(f"🔎 Verificación {MOTOR} vs VectorBT: {estado} "
          f"(máx. diferencia de retorno = {diferencia:.2e}, nº trades idéntico = {trades_ok})")

    gc.collect()            # ind_is se conserva en memoria para las pruebas de robustez (Celda 5)
except ErrorAmigable:
    raise
except Exception as error:
    raise ErrorAmigable(f"❌ Error en el motor de simulación: {error}\n"
                        "   Si es un error de memoria, reduce GRAN_MALLE_PARAMETROS o LOTE_MAX_COMBINACIONES.") from None

## 5️⃣ Embudo de robustez (GPU) y examen final Out-Of-Sample

Más combinaciones **no** dan mejores estrategias: cuantas más se prueban, más fácil es encontrar alguna que ganó **por casualidad**. Por eso la potencia de la GPU se dedica a **intentar romper** cada estrategia. Solo sobreviven las que resisten todo:

| Etapa | Qué comprueba | Por qué importa |
|---|---|---|
| 1. Filtros básicos | Retorno > 0, Profit Factor, nº de operaciones, drawdown | Descarta perdedoras y mediocres |
| 2. Consistencia temporal | El In-Sample se parte en 6 tramos: debe ganar en ≥ 4 | Evita estrategias que solo funcionaron en una racha |
| 3. Parámetros vecinos | Con parámetros parecidos (un paso arriba/abajo) también debe funcionar | Un pico aislado en el mapa de calor es sobreajuste |
| 4. Estrés de ejecución | Costes ×2 y entrada 1 vela más tarde | En real nunca se ejecuta perfecto |
| 5. Monte Carlo con ruido 🎲 | Cientos de versiones de la historia con ruido en los precios | Si depende de movimientos exactos del pasado, no se repetirá |
| 6. Prueba de azar 🎰 | La **malla completa** se repite sobre mercados barajados y sin tendencia: la estrategia debe superar lo que se consigue por pura suerte | Corrige el sesgo de probar miles de combinaciones (*Reality Check*) |
| 7. Examen final OOS | Datos nunca vistos + superar a **comprar y mantener** | La prueba definitiva |

- Las etapas 1-6 usan **solo** datos In-Sample; el Out-Of-Sample se mira **una sola vez**, al final.
- Las etapas 🎲 y 🎰 son las que más GPU consumen. **Para más rigor, sube `simulaciones_ruido` y `simulaciones_azar` en la Celda 2** (más tiempo de proceso = más certeza).
- Si el embudo se queda vacío es un resultado **honesto**: en ese activo/periodo la estrategia no tiene una ventaja fiable. Prueba otro activo, otra temporalidad u otra dirección.
- `p_valor_azar` ≈ probabilidad de que el resultado sea suerte. `sharpe_ruido_p10` = Sharpe en el 10% de escenarios más pesimistas.

In [ ]:
# ==============================================================================
# CELDA 5 · EMBUDO DE ROBUSTEZ (GPU) + EXAMEN FINAL OUT-OF-SAMPLE
# ==============================================================================
# Cada etapa intenta "romper" las estrategias. Solo pasan las que resisten todas:
#   1. Filtros básicos            → descarta perdedoras y mediocres
#   2. Consistencia temporal      → debe ganar en la mayoría de tramos del In-Sample
#   3. Estabilidad de parámetros  → sus parámetros vecinos también deben funcionar
#   4. Estrés de ejecución        → costes multiplicados y entrada con retraso
#   5. Monte Carlo con ruido      → cientos de versiones alternativas de la historia (GPU)
#   6. Prueba de azar             → la malla completa sobre precios sin patrones (GPU)
#   7. Examen final OOS           → datos nunca vistos + superar comprar y mantener
# Las etapas 1-6 usan SOLO datos In-Sample: el Out-Of-Sample se mira una única vez al final.
# ==============================================================================
import time
import warnings
from statistics import NormalDist

import matplotlib.pyplot as plt

estrategias_validas = None    # evita exportar resultados de una ejecución anterior
if globals().get("resultados_is") is None or globals().get("ind_is") is None:
    raise ErrorAmigable("❌ No hay resultados In-Sample: ejecuta primero la Celda 4 sin errores.")

R = ROBUSTEZ
SIM_RUIDO = int(R["simulaciones_ruido"])
SIM_AZAR = int(R["simulaciones_azar"])
if not KERNEL_CUDA_OK:   # En CPU estas pruebas serían muy lentas: se reducen
    SIM_RUIDO, SIM_AZAR = min(SIM_RUIDO, 20), min(SIM_AZAR, 20)
    print(f"🐢 Sin GPU: simulaciones reducidas a {SIM_RUIDO} (ruido) y {SIM_AZAR} (azar). "
          "La prueba de azar es menos precisa: usa la GPU para resultados fiables.")

COLUMNAS_CLON = ["retorno_total", "trades", "max_drawdown", "win_rate"]
embudo = [("Malla completa", len(resultados_is))]
T_INICIO = time.time()


def mostrar_embudo():
    tabla = pd.DataFrame(embudo, columns=["Etapa", "Estrategias que quedan"])
    tabla["% del total"] = (tabla["Estrategias que quedan"] / embudo[0][1]).map("{:.4%}".format)
    print("\n🔻 EMBUDO DE ROBUSTEZ")
    display(tabla.set_index("Etapa"))


def etapa(nombre, candidatas, consejo):
    """Registra cuántas quedan; si no queda ninguna, se detiene con una explicación."""
    embudo.append((nombre, len(candidatas)))
    print(f"   {'✅' if len(candidatas) else '❌'} {nombre:<40} → quedan {len(candidatas):>9,} "
          f"| {time.time() - T_INICIO:,.0f} s")
    if candidatas.empty:
        mostrar_embudo()
        raise ErrorAmigable(f"❌ Ninguna estrategia superó la etapa '{nombre}'.\n   👉 {consejo}")
    return candidatas


def pasa_filtros_basicos(met):
    fd = FILTROS_DESCARTE
    return ((met["retorno_total"] > fd["retorno_min"]) & (met["profit_factor"] >= fd["profit_factor_min"])
            & (met["trades"] >= fd["trades_min"]) & (met["trades"] <= fd["trades_max"])
            & (met["max_drawdown"] <= fd["drawdown_max"]) & met["sharpe"].notna()).to_numpy()


def sharpe_vecinos(resultados):
    """Sharpe mediano de los vecinos de cada combinación (±1 paso en cada parámetro).
    Se monta un cubo 4D (rápida × lenta × periodo RSI × umbral) y se desplaza en cada eje."""
    ejes = ["ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"]
    valores = [np.unique(resultados[e].to_numpy()) for e in ejes]
    posiciones = tuple(np.searchsorted(v, resultados[e].to_numpy()) for v, e in zip(valores, ejes))
    cubo = np.full([len(v) for v in valores], np.nan)                 # NaN = combinación no válida
    cubo[posiciones] = resultados["sharpe"].fillna(0.0).to_numpy()   # sin operaciones = Sharpe 0
    relleno = np.pad(cubo, 1, constant_values=np.nan)
    vecinos = []
    for eje in range(4):
        for paso in (-1, 1):
            corte = [slice(1, -1)] * 4
            corte[eje] = slice(1 + paso, relleno.shape[eje] - 1 + paso)
            vecinos.append(relleno[tuple(corte)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        mediana = np.nanmedian(np.stack(vecinos), axis=0)
    return pd.Series(mediana[posiciones], index=resultados.index)


def indicadores_con_ruido(base, sigma, semilla):
    """Versión alternativa de la historia: cada precio se mueve un poco al azar (en la GPU)."""
    aleatorio = xp.random.RandomState(semilla)
    n = base["close"].shape[0]
    close = base["close"] * (1 + sigma * aleatorio.standard_normal(n).astype(xp.float32))
    open_ = base["open"] * (1 + sigma * aleatorio.standard_normal(n).astype(xp.float32))
    return indicadores_desde_precios(open_, close)


def indicadores_sin_tendencia(base, semilla=None):
    """Precios SIN TENDENCIA: a cada movimiento se le resta el movimiento medio, de modo que
    'el mercado subió' deja de contar y solo puntúa saber cuándo entrar y salir.
    Con `semilla`, además se barajan los movimientos: un mercado SIN PATRONES (pura suerte)
    con la misma volatilidad que el real."""
    cierre = base["close"].astype(xp.float64)
    rendimientos = xp.diff(xp.log(cierre))
    rendimientos = rendimientos - rendimientos.mean()
    if semilla is not None:
        aleatorio = xp.random.RandomState(semilla)
        rendimientos = rendimientos[aleatorio.permutation(rendimientos.shape[0])]
    close = cierre[0] * xp.exp(xp.concatenate([xp.zeros(1), xp.cumsum(rendimientos)]))
    open_ = xp.concatenate([close[:1], close[:-1]])            # apertura = cierre anterior
    return indicadores_desde_precios(open_.astype(xp.float32), close.astype(xp.float32))


def progreso(i, total, etiqueta, t0):
    if total <= 10 or (i + 1) % max(1, total // 10) == 0 or i + 1 == total:
        print(f"      {etiqueta}: {i + 1:,}/{total:,} simulaciones | {time.time() - t0:,.0f} s", flush=True)


try:
    print(f"🧪 Embudo de robustez sobre {len(resultados_is):,} estrategias In-Sample ({MOTOR})")

    # ------------------------------------------------------------------------------
    # 1) Filtros básicos (máscaras vectorizadas sobre TODAS las combinaciones)
    # ------------------------------------------------------------------------------
    candidatas = etapa("1. Filtros básicos", resultados_is.loc[pasa_filtros_basicos(resultados_is)],
                       "Relaja FILTROS_DESCARTE, amplía la malla o cambia DIRECCION_OPERACION.")

    # ------------------------------------------------------------------------------
    # 2) Consistencia temporal (walk-forward): ganar en la mayoría de tramos del IS
    # ------------------------------------------------------------------------------
    limites = np.linspace(2, len(df_is), R["tramos_walk_forward"] + 1).astype(int)
    combos = malla.loc[candidatas.index]
    ganados = np.zeros(len(candidatas), dtype=np.int64)
    for a, b in zip(limites[:-1], limites[1:]):
        ganados += (backtest_masivo(ind_is, combos, inicio=a, fin=b)["retorno_total"].to_numpy() > 0)
    candidatas = candidatas.assign(tramos_ganadores=ganados)
    candidatas = etapa(f"2. Gana en ≥{R['tramos_ganadores_min']} de {R['tramos_walk_forward']} tramos",
                       candidatas[candidatas["tramos_ganadores"] >= R["tramos_ganadores_min"]],
                       "Baja ROBUSTEZ['tramos_ganadores_min'] o usa más historia.")

    # ------------------------------------------------------------------------------
    # 3) Estabilidad de parámetros: los vecinos también deben funcionar
    # ------------------------------------------------------------------------------
    candidatas = candidatas.assign(sharpe_vecinos=sharpe_vecinos(resultados_is).loc[candidatas.index])
    estables = candidatas["sharpe_vecinos"] >= np.maximum(0.0, R["fraccion_sharpe_vecinos"] * candidatas["sharpe"])
    candidatas = etapa("3. Parámetros vecinos estables", candidatas[estables],
                       "Baja ROBUSTEZ['fraccion_sharpe_vecinos'] o usa pasos más finos en la malla.")

    # Sin clones (mismas operaciones) y solo las mejores pasan al estrés
    candidatas = candidatas.sort_values(METRICA_RANKING, ascending=False)
    candidatas = candidatas.drop_duplicates(subset=COLUMNAS_CLON).head(int(R["candidatas_estres"]))
    embudo.append((f"   (sin clones, mejores {int(R['candidatas_estres']):,})", len(candidatas)))

    # ------------------------------------------------------------------------------
    # 4) Estrés de ejecución: costes ×N y entrada con retraso
    # ------------------------------------------------------------------------------
    combos = malla.loc[candidatas.index]
    extra = int(R["velas_retraso_extra"])
    m_costes = backtest_masivo(ind_is, combos, inicio=2, comision=COMISION_PCT * R["multiplicador_costes"])
    m_retraso = backtest_masivo(ind_is, combos, inicio=2 + extra, retraso=1 + extra)
    candidatas = candidatas.assign(retorno_costes_extra=m_costes["retorno_total"].to_numpy(),
                                   retorno_con_retraso=m_retraso["retorno_total"].to_numpy())
    resisten = (candidatas["retorno_costes_extra"] > 0) & (candidatas["retorno_con_retraso"] > 0)
    candidatas = etapa(f"4. Costes ×{R['multiplicador_costes']:g} y retraso +{extra} vela",
                       candidatas[resisten], "Las estrategias dependen de una ejecución perfecta: "
                       "prueba temporalidades mayores (H1/H4) o baja el estrés.")

    # ------------------------------------------------------------------------------
    # 5) Monte Carlo con ruido en los precios (GPU intensiva)
    # ------------------------------------------------------------------------------
    combos = malla.loc[candidatas.index]
    cierre_is = df_is["close"].to_numpy(np.float64)
    sigma = float(np.std(np.diff(np.log(cierre_is)))) * R["ruido_fraccion_volatilidad"]
    print(f"   🎲 Monte Carlo: {SIM_RUIDO:,} historias con ruido × {len(combos):,} estrategias "
          f"× {len(df_is):,} velas")
    retornos_ruido = np.empty((SIM_RUIDO, len(combos)))
    sharpes_ruido = np.empty((SIM_RUIDO, len(combos)))
    t0 = time.time()
    for i in range(SIM_RUIDO):
        m = backtest_masivo(indicadores_con_ruido(ind_is, sigma, 1_000 + i), combos, inicio=2)
        retornos_ruido[i] = m["retorno_total"].to_numpy()
        sharpes_ruido[i] = m["sharpe"].to_numpy()
        progreso(i, SIM_RUIDO, "ruido", t0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        candidatas = candidatas.assign(
            ruido_pct_ganadoras=(retornos_ruido > 0).mean(axis=0),
            sharpe_ruido_mediana=np.nanmedian(sharpes_ruido, axis=0),
            sharpe_ruido_p10=np.nanpercentile(sharpes_ruido, 10, axis=0),   # escenario pesimista
        )
    candidatas = etapa(f"5. Gana en ≥{R['ruido_ganadoras_min']:.0%} de historias con ruido",
                       candidatas[candidatas["ruido_pct_ganadoras"] >= R["ruido_ganadoras_min"]],
                       "Baja ROBUSTEZ['ruido_ganadoras_min'] o 'ruido_fraccion_volatilidad'.")

    # ------------------------------------------------------------------------------
    # 6) Prueba de azar (Reality Check de White, con la corrección de tendencia de Aronson):
    #    ¿lo encontrado supera lo que da la suerte al probar TANTAS combinaciones?
    #    • Se quita la tendencia a los precios reales y se mide el Sharpe de cada candidata.
    #    • La malla COMPLETA se repite sobre precios sin tendencia y BARAJADOS (sin patrones).
    #      El mejor Sharpe de cada repetición es "lo que se consigue por pura suerte".
    #    • p-valor = proporción de mercados aleatorios cuyo mejor resultado iguala o supera a la
    #      candidata. Sin quitar la tendencia, en un activo que sube cualquier estrategia de
    #      compras parecería buena y la comparación no sería justa.
    # ------------------------------------------------------------------------------
    m_real = backtest_masivo(indicadores_sin_tendencia(ind_is), malla.loc[candidatas.index], inicio=2)
    candidatas = candidatas.assign(sharpe_sin_tendencia=m_real["sharpe"].fillna(0.0).to_numpy())
    print(f"   🎰 Prueba de azar: {SIM_AZAR:,} mercados sin patrones × {len(malla):,} estrategias "
          f"× {len(df_is):,} velas")
    maximos_azar = np.zeros(SIM_AZAR)
    t0 = time.time()
    for i in range(SIM_AZAR):
        m = backtest_masivo(indicadores_sin_tendencia(ind_is, semilla=5_000 + i), malla, inicio=2)
        validas = pasa_filtros_basicos(m)
        maximos_azar[i] = m["sharpe"].to_numpy()[validas].max() if validas.any() else 0.0
        progreso(i, SIM_AZAR, "azar", t0)
    # Umbral de suerte: percentil `confianza` de los máximos. Con pocas simulaciones se usa
    # además una estimación normal (media + z·desviación), y se toma la más exigente.
    z = NormalDist().inv_cdf(R["confianza_azar"])
    sharpe_suerte = float(np.quantile(maximos_azar, R["confianza_azar"]))
    if SIM_AZAR > 1:
        sharpe_suerte = max(sharpe_suerte, float(maximos_azar.mean() + z * maximos_azar.std(ddof=1)))
    print(f"      Mejor Sharpe por azar (sin tendencia): mediana {np.median(maximos_azar):.2f} | "
          f"umbral de suerte al {R['confianza_azar']:.0%} = {sharpe_suerte:.2f} | "
          f"mejor candidata real sin tendencia: {candidatas['sharpe_sin_tendencia'].max():.2f}")
    superan = (maximos_azar[None, :] >= candidatas["sharpe_sin_tendencia"].to_numpy()[:, None]).sum(axis=1)
    candidatas = candidatas.assign(p_valor_azar=(1 + superan) / (1 + SIM_AZAR))   # informativo
    candidatas = etapa(f"6. Supera al azar (Sharpe > {sharpe_suerte:.2f})",
                       candidatas[candidatas["sharpe_sin_tendencia"] > sharpe_suerte],
                       "Ninguna estrategia supera lo que se obtiene por suerte con esta malla: "
                       "prueba otro activo, otra temporalidad, otra dirección o una malla más pequeña.")

    # ------------------------------------------------------------------------------
    # 7) Examen final: VectorBT Portfolio.from_signals en IS y en el OOS nunca visto
    # ------------------------------------------------------------------------------
    top_is = candidatas.sort_values("sharpe_ruido_p10", ascending=False).head(TOP_N)
    combos_top = malla.loc[top_is.index]
    ind_is_np = {k: a_numpy(v) for k, v in ind_is.items()}
    met_is_vbt, pf_is = simular_vectorbt(df_is, ind_is_np, combos_top, inicio=2)
    # OOS: indicadores calculados SOLO con datos OOS; la purga (calentamiento) no se opera
    ind_oos_np = {k: a_numpy(v) for k, v in construir_indicadores(df_oos_completo).items()}
    met_oos_vbt, pf_oos = simular_vectorbt(df_oos_completo, ind_oos_np, combos_top, inicio=PERIODO_PURGA)

    # Referencia: comprar y mantener (o vender y mantener) en el mismo periodo OOS
    rend_ref = df_oos["close"].astype(np.float64).pct_change().dropna()
    rend_ref = rend_ref if ES_COMPRA else -rend_ref
    sharpe_ref = float(rend_ref.mean() / rend_ref.std() * np.sqrt(BARRAS_POR_ANIO))
    retorno_ref = float((1 + rend_ref).prod() - 1)
    nombre_ref = "Comprar y mantener" if ES_COMPRA else "Vender y mantener"
    print(f"\n📏 Referencia OOS · {nombre_ref}: retorno {retorno_ref:.2%} | Sharpe {sharpe_ref:.2f}")

    fo = FILTROS_OOS
    columnas_robustez = ["tramos_ganadores", "sharpe_vecinos", "retorno_costes_extra", "retorno_con_retraso",
                         "ruido_pct_ganadoras", "sharpe_ruido_mediana", "sharpe_ruido_p10",
                         "sharpe_sin_tendencia", "p_valor_azar"]
    comparativa = pd.concat([
        combos_top[["ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"]],
        top_is[columnas_robustez],
        met_is_vbt.add_suffix("_IS"),
        met_oos_vbt.add_suffix("_OOS"),
    ], axis=1)
    comparativa["retorno_referencia_OOS"] = retorno_ref
    comparativa["sharpe_referencia_OOS"] = sharpe_ref
    comparativa["supera_OOS"] = (
        (comparativa["retorno_total_OOS"] > fo["retorno_min"])
        & (comparativa["profit_factor_OOS"] >= fo["profit_factor_min"])
        & (comparativa["trades_OOS"] >= fo["trades_min"])
    )
    if R["exigir_superar_buy_hold"]:
        comparativa["supera_OOS"] &= comparativa["sharpe_OOS"] > sharpe_ref
    comparativa["robustez_sharpe"] = comparativa["sharpe_OOS"] / comparativa["sharpe_IS"]
    comparativa.insert(0, "ranking_robustez", np.arange(1, len(comparativa) + 1))
    estrategias_validas = comparativa[comparativa["supera_OOS"]].copy()
    embudo.append((f"7. Examen final OOS (de las {len(comparativa)} más robustas)", len(estrategias_validas)))

    mostrar_embudo()
    print(f"⏱️ Tiempo total del embudo: {time.time() - T_INICIO:,.0f} s")

    if estrategias_validas.empty:
        print("\n⚠️ Ninguna estrategia robusta superó el examen final Out-Of-Sample"
              + (f" ni la referencia '{nombre_ref}'" if R["exigir_superar_buy_hold"] else "") + ".\n"
              "   Es un resultado honesto: en este activo/periodo la estrategia no aporta una ventaja fiable.")
    else:
        columnas_tabla = ["ranking_robustez", "ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral",
                          "retorno_total_IS", "retorno_total_OOS", "sharpe_IS", "sharpe_OOS",
                          "sharpe_ruido_p10", "p_valor_azar", "tramos_ganadores", "ruido_pct_ganadoras",
                          "profit_factor_IS", "profit_factor_OOS", "max_drawdown_IS", "max_drawdown_OOS",
                          "trades_IS", "trades_OOS", "robustez_sharpe"]
        formato = {c: "{:.2%}" for c in columnas_tabla if c.startswith(("retorno", "max_drawdown", "ruido_pct"))}
        formato.update({c: "{:.2f}" for c in columnas_tabla if c.startswith(("profit", "sharpe", "robustez"))})
        formato.update({"rsi_umbral": "{:.0f}", "p_valor_azar": "{:.3f}"})
        try:
            display(estrategias_validas[columnas_tabla].style.format(formato)
                    .background_gradient(subset=["sharpe_OOS"], cmap="RdYlGn").hide(axis="index"))
        except Exception:   # Sin jinja2 → tabla simple
            display(estrategias_validas[columnas_tabla].round(4))

    # ------------------------------------------------------------------------------
    # Gráficos: mapa de calor de parámetros y curva de capital de la mejor estrategia
    # ------------------------------------------------------------------------------
    if MOSTRAR_GRAFICOS:
        metrica_mapa = METRICA_RANKING
        mapa = resultados_is.pivot_table(index="ma_lenta", columns="ma_rapida",
                                         values=metrica_mapa, aggfunc="max")
        mapa = mapa.replace([np.inf, -np.inf], np.nan)
        fig, ax = plt.subplots(figsize=(14, 7))
        lim = np.nanpercentile(np.abs(mapa.to_numpy()), 98) if np.isfinite(mapa.to_numpy()).any() else 1
        im = ax.imshow(mapa.to_numpy(), aspect="auto", origin="lower", cmap="RdYlGn", vmin=-lim, vmax=lim,
                       extent=[mapa.columns.min(), mapa.columns.max(), mapa.index.min(), mapa.index.max()])
        fig.colorbar(im, ax=ax, label=f"{metrica_mapa} (mejor valor por par de medias)")
        if not estrategias_validas.empty:
            ax.scatter(estrategias_validas["ma_rapida"], estrategias_validas["ma_lenta"], s=40,
                       facecolors="none", edgecolors="black", linewidths=1.2, label="Estrategias robustas")
            ax.legend(loc="upper right")
        ax.set_xlabel("Periodo MA rápida")
        ax.set_ylabel("Periodo MA lenta")
        ax.set_title(f"Mapa de calor In-Sample · {NOMBRE_ACTIVO} · {DIRECCION_OPERACION}\n"
                     "Zonas verdes amplias = parámetros robustos; puntos aislados = posible sobreajuste")
        plt.show()

        if not estrategias_validas.empty:
            mejor = estrategias_validas.index[0]
            pos = combos_top.index.get_loc(mejor)
            curva_is = pf_is.value().iloc[:, pos]
            # El OOS empieza con CAPITAL_INICIAL: se reescala para continuar donde acabó el IS
            curva_oos = pf_oos.value().iloc[:, pos] / CAPITAL_INICIAL * curva_is.iloc[-1]
            referencia = (1 + rend_ref).cumprod() * curva_is.iloc[-1]
            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(curva_is.index, curva_is.to_numpy(), color="#2563eb", label="In-Sample")
            ax.plot(curva_oos.index, curva_oos.to_numpy(), color="#16a34a", label="Out-Of-Sample")
            ax.plot(referencia.index, referencia.to_numpy(), color="#9ca3af", linewidth=1,
                    label=f"{nombre_ref} (OOS)")
            ax.axvline(df_oos.index[0], color="gray", linestyle="--", linewidth=1)
            fila = estrategias_validas.loc[mejor]
            if fila.rsi_umbral >= 100:
                filtro = "sin filtro RSI"
            elif ES_COMPRA:
                filtro = f"RSI{fila.rsi_periodo:.0f} < {fila.rsi_umbral:.0f}"
            else:
                filtro = f"RSI{fila.rsi_periodo:.0f} > {100 - fila.rsi_umbral:.0f}"
            ax.set_title(f"Curva de capital · {NOMBRE_ACTIVO} · {DIRECCION_OPERACION} · "
                         f"MA {fila.ma_rapida:.0f}/{fila.ma_lenta:.0f} · {filtro}")
            ax.set_ylabel("Capital")
            if max(curva_is.max(), curva_oos.max()) / CAPITAL_INICIAL > 10:
                ax.set_yscale("log")             # escala logarítmica si el capital crece mucho
                ax.set_ylabel("Capital (escala logarítmica)")
            ax.legend()
            ax.grid(alpha=0.3)
            plt.show()

    del ind_is_np, ind_oos_np
    gc.collect()
except ErrorAmigable:
    raise
except Exception as error:
    raise ErrorAmigable(f"❌ Error en el embudo de robustez o la validación OOS: {error}") from None

## 6️⃣ Exportación de resultados a Google Drive

Guarda **solo** las estrategias que superaron el filtro In-Sample **y** la validación Out-Of-Sample en:

`/content/drive/MyDrive/activos/resultados_[NOMBRE]_[DIRECCION].csv` (ej. `resultados_XAUUSD_H1_COMPRAS_SOLO.csv`)

In [ ]:
# ==============================================================================
# CELDA 6 · EXPORTACIÓN DE RESULTADOS A GOOGLE DRIVE
# ==============================================================================
try:
    if globals().get("estrategias_validas") is None:
        raise ErrorAmigable("❌ No hay resultados validados: ejecuta primero la Celda 5 sin errores.")
    if estrategias_validas.empty:
        print("ℹ️ No hay estrategias que hayan superado los filtros IS y la validación OOS: "
              "no se genera archivo.")
    else:
        nombre_base = NOMBRE_ACTIVO
        ruta_salida = os.path.join(RUTA_ACTIVOS, f"resultados_{nombre_base}_{DIRECCION_OPERACION}.csv")
        exportar = estrategias_validas.drop(columns=["supera_OOS"]).copy()
        exportar.insert(0, "activo", nombre_base)
        exportar.insert(1, "direccion", DIRECCION_OPERACION)
        exportar["comision_pct"] = COMISION_PCT
        exportar["inicio_IS"] = df_is.index[0]
        exportar["fin_IS"] = df_is.index[-1]
        exportar["inicio_OOS"] = df_oos.index[0]
        exportar["fin_OOS"] = df_oos.index[-1]
        exportar = exportar.replace([np.inf, -np.inf], np.nan).round(6)
        exportar.to_csv(ruta_salida, index=False, encoding="utf-8")
        print(f"💾 {len(exportar)} estrategias robustas guardadas en:\n   {ruta_salida}")
        display(exportar.head(10))
except ErrorAmigable:
    raise
except Exception as error:
    raise ErrorAmigable(f"❌ No se pudo guardar el CSV en Google Drive: {error}\n"
                     "   Comprueba que Drive sigue montado (vuelve a ejecutar la Celda 1).") from None